# Options order flow and intraday volatility
## Extended walkthrough

This notebook checks the six-asset results in Chapter 4 against the aggregate files published in the repository and regenerates Figures 5 to 8. It fits no model and reads no licensed data. Select **Runtime → Restart session and run all**. Allow 30 minutes to read the walkthrough; the calculations themselves take about a minute. The final Claims ledger compares the report's printed values with the calculations and saved statistics, followed by an input and execution receipt.

Each step verifies its inputs, computes or reads the relevant result, and closes with a reading tied to the report. The separate model-selection and eight-asset extensions in Tables 15 and 16 use other result packages and are outside this companion. The [Quick verification](https://colab.research.google.com/drive/1k9eTbP5fv7LlWVEyP1Kydbonzif52rZC) checks only the headline tables.

Inputs are 15 CSV files and one JSON file from the public repository release cited in the report. Their fingerprints are checked in Step 1 and listed in the receipt. The report locations checked here are Tables 3, 6 to 14, 17 and 19, plus Section 4.10.

### Terms used below

B0 contains price history; B1 adds **option state**; B2 adds the **mixed flow block**. Their cumulative predictor counts are 29, 69 and 138. H1 compares B1 with B0; H2 compares B2 with B1, within the same model family. Linear denotes the winsorised ridge model on log variance; Trees denotes LightGBM.

RV15 is realised variance over the next 15 minutes; RV30 and RV5 use 30 and 5 minutes. QLIKE is dimensionless: for observed variance y > 0 and forecast f > 0, L = y/f − log(y/f) − 1. Lower loss is better. A paired difference is baseline loss minus expanded-set loss; reduction (%) = 100 × mean difference / mean baseline loss. Loss reductions are forecast-error reductions, not returns.

A **session** is a market day; an **origin** is an eligible forecast time for an asset. The development window contains 419 sessions and 160,832 origins; the final window contains 25 sessions. In source columns, `primary` means development and `confirmation` means final; `log_ridge_harq` means Linear and `lightgbm_qlike` means Trees. `B1_over_B0` and `B2_over_B1` identify comparisons, not loss ratios. These definitions follow Table 4 and Table 22.


## Step 1: Verify inputs

The embedded package supplies every input. The checks validate the archive, the unique file names and every file fingerprint before any calculation.


In [1]:
# @title Step 1: Verify inputs
# Start an empty completion record for this run.
EXECUTED_CELLS = []
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import html
# Pin the public input bytes and the second implementation's expected digest.
SOURCE_COMMIT = '9b63302e2a9d1c8c43fb8755628f0e3dfc48ec03'
PUBLIC_SOURCE_COMMIT = '9b63302e2a9d1c8c43fb8755628f0e3dfc48ec03'
PUBLIC_HASHES = {'artifacts/rp4_v4_b4/primary_statistics.csv': '5c215fc38344839ecedea7b27f69efcb85fbefea84f1367d06229cb7407902d4', 'artifacts/rp4_v4_b2_rv15/session_losses.csv': '28174eb5a4f032d0b31b6a7b65e76e05710168278419d87bf31a92ebec2d9ac1', 'artifacts/rp4_v4_b4/coverage.csv': '67941d02647ae329fcc704a0b109949ab4312cb6915006302186b21770e853ef', 'artifacts/rp4_v4_b4/robustness.csv': '09925f9cefe4fcf2c19f4a915ad2994c125bbafdb94dfd332cc821ce1f180acb', 'artifacts/rp4_robustness_public_v1/reference_qlike.csv': 'af976a11074c144f024703c3d149a71103a6fde6bee11f7e43da64aeb19aacda', 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv': '3e59dcbab5760464b0abf41ca2f748016e7d42a51542c195c3cee6687a2e862e', 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv': '66ba437b3878bb4b2c5241863236b0eaf59712eaf3be8a63bbe40a1baca56f88', 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv': '65cce2e2c23602249ce2eb1f1f00917729580689d14900134a59ded89bc0d698', 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv': '976c95b896f9f2a836ee53a985e8462f7a6cfe2139690d66e264367f1ab78df4', 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv': '3d5a8decdf511c04b225e49894b23f54996ea0ebfc6476f8d3ed4357469098c1', 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv': '3e8b96c33e63d772f786555dfb948a170c970a49e3c490b22794570d0ac9ac8c', 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv': '58a8ab77312256ab7b198f1b685b576e800d7619d6138476d16ba4c1e4b36494', 'artifacts/rp4_v4_b2_rv15/summary.json': '451e53869bb5f5058a635302bb08ad920ce11b8771fd2c6780545255b1ac25f4', 'artifacts/rp4_v4_b4/regime_secondary.csv': 'eb1fd165dd03f5c24efb717afdae7a30e04898dcd6fc53cb3172c1a2d81b03c7'}
BUNDLE_SHA256 = 'f3f82f184e585574b950ad4c65301fe57ae7e9853271275fbb875f083911b263'
CROSSCHECK_EXPECTED_DIGEST = 'db0fad77a1ce7b2bf2cfd33d4ffb47cef119219834ab76cd2ac02d5eea8c4cf2'
BUNDLE_BASE64 = '...'
# Validate the encoding and the complete archive before reading any input.
raw_bundle = base64.b64decode(BUNDLE_BASE64, validate=True)
assert hashlib.sha256(raw_bundle).hexdigest() == BUNDLE_SHA256
public_archive = zipfile.ZipFile(io.BytesIO(raw_bundle))
assert len(public_archive.namelist()) == len(set(public_archive.namelist())) == 14
assert set(public_archive.namelist()) == set(PUBLIC_HASHES)
# Verify the two additional cutoff files without changing the original package checks.
SUPPLEMENTARY_SHA256 = '5bfd20135b7a7b3d9d41808c23a1cbf57e3199936a586d9bc96e66367f20aa9f'
SUPPLEMENTARY_BASE64 = '...'
supplementary_raw = base64.b64decode(SUPPLEMENTARY_BASE64, validate=True)
assert hashlib.sha256(supplementary_raw).hexdigest() == SUPPLEMENTARY_SHA256
supplementary_archive = zipfile.ZipFile(io.BytesIO(supplementary_raw))
supplementary_hashes = {'artifacts/rp4_robustness_public_v1/pit_60_session_losses.csv': '96598b958628074403c82835dcaa6517d9c86dadd650e444ac11b6854a61b5f6', 'artifacts/rp4_robustness_public_v1/pit_300_session_losses.csv': '553d035a0b087dbc0e48a1d8bf9c3bfdc4e2a725b7fef9c3e01182ba48997c56'}
assert len(supplementary_archive.namelist()) == len(set(supplementary_archive.namelist())) == 2
assert set(supplementary_archive.namelist()) == set(supplementary_hashes)
assert not set(PUBLIC_HASHES).intersection(supplementary_hashes)
PUBLIC_HASHES.update(supplementary_hashes)
# Every read checks the file fingerprint, including subsequent reads.
def fetch_bytes(path):
    raw = (supplementary_archive if path in supplementary_hashes else public_archive).read(path)
    assert hashlib.sha256(raw).hexdigest() == PUBLIC_HASHES[path], path
    return raw
for path in PUBLIC_HASHES:
    fetch_bytes(path)
manifest = pd.DataFrame([{'File':p.rsplit('/',1)[-1], 'Integrity':'Verified'} for p in PUBLIC_HASHES])
display(HTML('<h3>Inputs ready: 15 CSV files and one JSON file verified</h3>'))
display(manifest.style.hide(axis='index'))
hash_detail=''.join('<p><b>'+html.escape(p)+'</b><br><code>'+sha+'</code></p>' for p,sha in PUBLIC_HASHES.items())
display(HTML('<details><summary>Input fingerprints and source reference</summary><p>'+PUBLIC_SOURCE_COMMIT+'</p>'+hash_detail+'</details>'))
# Reject an empty table rather than continuing with missing results.
def read_public_csv(path):
    frame = pd.read_csv(io.BytesIO(fetch_bytes(path)))
    assert not frame.empty, path
    return frame
statistics = read_public_csv('artifacts/rp4_v4_b4/primary_statistics.csv')
losses = read_public_csv('artifacts/rp4_v4_b2_rv15/session_losses.csv')
summary = json.loads(fetch_bytes('artifacts/rp4_v4_b2_rv15/summary.json'))
# Labels are separate from the source identifiers used in every calculation.
families = {'log_ridge_harq':'Linear','lightgbm_qlike':'Trees'}

# Record that this step completed.
EXECUTED_CELLS.append('step_01_verify_inputs')


File,Integrity
primary_statistics.csv,Verified
session_losses.csv,Verified
coverage.csv,Verified
robustness.csv,Verified
reference_qlike.csv,Verified
reference_contrasts.csv,Verified
secondary_metrics.csv,Verified
pit_60_contrasts.csv,Verified
pit_300_contrasts.csv,Verified
placebo_log_ridge_harq_rv15_draws.csv,Verified


### Reading

These are the aggregate inputs described in Section 3.10 and used in Table 7. All input fingerprints must match before the first result is calculated.


## Three example sessions

Verify the session-loss file, then inspect its first, middle and last dates. Each row below subtracts the Linear B2 loss from the B1 loss on the same session.


In [2]:
# @title Three example sessions
# Select the first, middle and last sessions in chronological order.
chronological = losses.sort_values('session_date').reset_index(drop=True)
positions = [0, len(chronological)//2, len(chronological)-1]
example = chronological.iloc[positions][['session_date', 'loss__log_ridge_harq__B1', 'loss__log_ridge_harq__B2']].copy()
example.columns = ['session', 'QLIKE B1', 'QLIKE B2']
# Subtract paired losses for the same session.
example['delta B1 − B2'] = example['QLIKE B1'] - example['QLIKE B2']
display(example.style.format({'QLIKE B1':'{:.9f}', 'QLIKE B2':'{:.9f}', 'delta B1 − B2':'{:+.9f}'}))
# Show the operands of the first paired difference.
first = example.iloc[0]
print(f"First session: {first['QLIKE B1']:.9f} − {first['QLIKE B2']:.9f} = {first['delta B1 − B2']:+.9f} QLIKE.")
print(f"CSV date range: {chronological.session_date.iloc[0]} to {chronological.session_date.iloc[-1]}; {len(chronological)} sessions.")

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_02_three_example_sessions')


,session,QLIKE B1,QLIKE B2,delta B1 − B2
0,2024-10-28,0.149004844,0.140765198,+0.008239646
209,2025-09-30,0.141206386,0.144102384,-0.002895998
418,2026-07-31,0.154928260,0.151413468,+0.003514792


First session: 0.149004844 − 0.140765198 = +0.008239646 QLIKE.
CSV date range: 2024-10-28 to 2026-07-31; 419 sessions.


### Reading

The three dates illustrate the subtraction used in the full calculation. Table 7 averages paired differences across all 419 sessions, with equal weight per session.


## The four primary contrasts

Verify the session counts, unique dates and finite losses. The calculation uses the already aggregated session losses; applying QLIKE to averaged targets and forecasts would change the estimand. The original checks require agreement within 1e-12 QLIKE and 1e-10 percentage points.


In [3]:
# @title The four primary contrasts
# Use math.fsum for accurate summation and Matplotlib for the figures.
import math

import matplotlib
import matplotlib.pyplot as plt


# Compare paired session losses with one published contrast.
def compare_saved(frame, row):
    """Reconcile one published contrast with paired session losses."""
    # Split the expanded and baseline information sets within each model family.
    expanded, baseline = row.contrast.split("_over_")
    base = frame[f"loss__{row.family}__{baseline}"]
    extra = frame[f"loss__{row.family}__{expanded}"]
    # Reject nonfinite losses and duplicate dates before taking means.
    if not all(math.isfinite(x) for x in [*base, *extra]):
        raise ValueError("Nonfinite losses.")
    if frame.session_date.nunique() != len(frame):
        raise ValueError("Duplicate sessions.")
    # Weight sessions equally; a positive difference favors the expanded model.
    mean_base = math.fsum(base) / len(base)
    delta = math.fsum(base - extra) / len(base)
    # Express the difference as a percentage of the mean baseline loss.
    percent = 100 * delta / mean_base
    # Check the count, mean, difference and percentage against explicit tolerances.
    assert len(frame) == row.N_sessions
    assert abs(mean_base - row.baseline_loss) <= 1e-12
    assert abs(delta - row.estimate) <= 1e-12
    assert abs(percent - row.qlike_reduction_percent) <= 1e-10
    # Return the computed values and their differences from the published values.
    return {"family": row.family, "contrast": row.contrast,
            "sessions": len(frame), "delta_QLIKE": delta,
            "reduction_percent": percent, "delta_error": delta - row.estimate,
            "check": "PASS: aggregates"}


# Select RV15 in the development window.
selected = statistics.loc[
    (statistics.horizon_minutes == 15) & (statistics.window == "primary")
]
# Recompute both contrasts in both model families.
checks = pd.DataFrame(compare_saved(losses, row)
                      for row in selected.itertuples())
assert len(checks) == 4
print("Four RV15 contrasts: arithmetic checks passed")
display(checks)

# Sort session dates before drawing the cumulative series.
ordered = losses.sort_values("session_date")
dates = pd.to_datetime(ordered.session_date)
# Plot paired losses and cumulative QLIKE differences in separate panels.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
for family, label, color in [
    ("log_ridge_harq", "Linear", "#0072B2"),
    ("lightgbm_qlike", "Trees", "#D55E00"),
]:
    # Use the same dates for B1 and B2; each point is one session.
    base = ordered[f"loss__{family}__B1"]
    extra = ordered[f"loss__{family}__B2"]
    axes[0].scatter(base, extra, s=18, alpha=0.5, color=color, label=label)
    axes[1].plot(dates, (base - extra).cumsum(), color=color, label=label)
# Draw the equal-loss diagonal; points below it favor B2.
limit = max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])
axes[0].plot([0, limit], [0, limit], "k--", linewidth=1, label="Equal loss")
axes[0].set(xlabel="B1: session QLIKE", ylabel="B2: session QLIKE",
            title="419 sessions: points below the diagonal favor B2",
            xlim=(0, limit), ylim=(0, limit), aspect="equal")
# Draw zero as the boundary between positive and negative cumulative differences.
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="Session date", ylabel="Cumulative QLIKE difference (B1 − B2)",
            title="Cumulative difference; positive favors B2")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.2)
axes[1].tick_params(axis="x", rotation=30)
plt.show()
print("Matplotlib:", matplotlib.__version__)

# Check the reported percentages rounded to three decimal places.
expected_rounded = {('log_ridge_harq','B1_over_B0'):0.880,
                    ('log_ridge_harq','B2_over_B1'):0.623,
                    ('lightgbm_qlike','B1_over_B0'):1.170,
                    ('lightgbm_qlike','B2_over_B1'):-0.115}
# Match each percentage to its model family and contrast.
for row in checks.to_dict('records'):
    assert round(row['reduction_percent'],3) == expected_rounded[(row['family'],row['contrast'])]
print('PASS: 0.880 / 0.623 / 1.170 / −0.115 %; primary_statistics.csv checks passed')

# Record completion after all checks and figures finish.

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_03_the_four_primary_contrasts')


Four RV15 contrasts: arithmetic checks passed


,family,contrast,sessions,delta_QLIKE,reduction_percent,delta_error,check
0,log_ridge_harq,B1_over_B0,419,0.001616,0.880055,2.580401e-17,PASS: aggregates
1,log_ridge_harq,B2_over_B1,419,0.001134,0.622794,5.746272e-17,PASS: aggregates
2,lightgbm_qlike,B1_over_B0,419,0.002197,1.169969,3.989864e-17,PASS: aggregates
3,lightgbm_qlike,B2_over_B1,419,-0.000213,-0.114739,-6.155558e-17,PASS: aggregates


Matplotlib: 3.11.2
PASS: 0.880 / 0.623 / 1.170 / −0.115 %; primary_statistics.csv checks passed


### Reading

Option state reduces loss by 0.880% in Linear and 1.170% in Trees. The mixed flow block adds 0.623% in Linear and −0.115% in Trees, so the increment depends on the family. These are the four contrasts in Table 7; the cumulative panel represents Figure 8.


## The final window

Verify the eight RV15 rows spanning both windows, then read their saved intervals and decision states. H2 is tested formally only after H1 rejects. Missing formal H2 probabilities remain missing; the report also gives nominal diagnostics.


In [4]:
# @title The final window
# Read the stored inference columns.
inference_columns = ['window','family','contrast','N_sessions','qlike_reduction_percent','ci_low','ci_high','p_for_decision','hypothesis_status']
rv15_inference = statistics.loc[statistics.horizon_minutes.eq(15), inference_columns].copy()
assert len(rv15_inference) == 8
# Leave unavailable formal p-values marked as not tested.
display(rv15_inference.style.format({'qlike_reduction_percent':'{:+.3f}', 'ci_low':'{:+.6f}', 'ci_high':'{:+.6f}', 'p_for_decision':'{:.4f}'}, na_rep='Not tested'))
# Check the H1-to-H2 testing sequence in the final window.
final_rows = rv15_inference[rv15_inference.window.eq('confirmation')]
assert set(final_rows[final_rows.contrast.eq('B1_over_B0')].hypothesis_status) == {'NOT_REJECTED'}
assert set(final_rows[final_rows.contrast.eq('B2_over_B1')].hypothesis_status) == {'NOT_TESTED'}

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_04_the_final_window')


,window,family,contrast,N_sessions,qlike_reduction_percent,ci_low,ci_high,p_for_decision,hypothesis_status
8,primary,log_ridge_harq,B1_over_B0,419,+0.880,+0.000253,+0.003454,0.0390,REJECTED
9,primary,log_ridge_harq,B2_over_B1,419,+0.623,+0.000335,+0.001932,0.0032,REJECTED
10,primary,lightgbm_qlike,B1_over_B0,419,+1.170,+0.000611,+0.004103,0.0135,REJECTED
11,primary,lightgbm_qlike,B2_over_B1,419,-0.115,-0.002044,+0.001104,0.6280,NOT_REJECTED
12,confirmation,log_ridge_harq,B1_over_B0,25,+0.174,-0.002209,+0.002878,0.3908,NOT_REJECTED
13,confirmation,log_ridge_harq,B2_over_B1,25,+1.997,-0.001109,+0.014742,Not tested,NOT_TESTED
14,confirmation,lightgbm_qlike,B1_over_B0,25,+3.163,+0.001644,+0.016807,0.0568,NOT_REJECTED
15,confirmation,lightgbm_qlike,B2_over_B1,25,-0.992,-0.009072,+0.002195,Not tested,NOT_TESTED


### Reading

Neither family rejects H1 in the final window, so H2 remains untested under the registered sequence. The Trees H1 probability is 0.0568. Table 13 reports this result; its positive point estimates do not confirm the full sequence.


## Coverage of the development sample

Verify coverage and subset files, then inspect their dimensions and the counts retained for the development window. Scheduled and eligible origins use the same sampling frame.


In [5]:
# @title Coverage of the development sample
# Read coverage and subset summaries from the verified CSV bytes.
coverage = read_public_csv("artifacts/rp4_v4_b4/coverage.csv")
robustness = read_public_csv("artifacts/rp4_v4_b4/robustness.csv")
# Count missing values in the aggregate tables.
print("Missing values in the aggregate CSVs:")
# Display missing-value counts for each column.
for name, frame in [("losses", losses), ("coverage", coverage)]:
    display(frame.isna().sum().rename(name).to_frame())
# List the available subsets before selecting rows.
print("Available subsets:", sorted(robustness.subset.unique()))

# Record completion after all checks and displays finish.

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_05_coverage_of_the_development_sample')


Missing values in the aggregate CSVs:


,losses
session_date,0
forecast__log_ridge_harq__B0,0
loss__log_ridge_harq__B0,0
forecast__log_ridge_harq__B1,0
loss__log_ridge_harq__B1,0
forecast__log_ridge_harq__B2,0
loss__log_ridge_harq__B2,0
forecast__lightgbm_qlike__B0,0
loss__lightgbm_qlike__B0,0
forecast__lightgbm_qlike__B1,0


,coverage
horizon_minutes,0
window,0
asset,0
eligible_rows,0
failed_quality_gate,0
incomplete_mandatory_predictors,0
invalid_target,0
scheduled_rows,0


Available subsets: ['asset_AAPL', 'asset_AMZN', 'asset_META', 'asset_MSFT', 'asset_NVDA', 'asset_TSLA', 'chronological_block_1', 'chronological_block_2', 'chronological_block_3', 'last30sessions', 'leave_block_1_out', 'leave_block_2_out', 'leave_block_3_out']


### Reading

The six assets contribute 160,832 eligible origins. The difference between scheduled and eligible counts is an exclusion count; individual exclusion reasons may overlap. Table 3 reports the asset totals and Figure 5 shows eligibility and known-empty windows.


## Session differences, assets and chronological blocks

Compute paired session differences, their descriptive quantiles and the proportion that is positive. Then read the saved asset and chronological-block contrasts. The date table reconstructs the producer's three consecutive blocks: the first n//3 dates, the next n//3, and the remainder.


In [6]:
# @title Session differences, assets and chronological blocks
# Select RV15 coverage in the development window.
cov = coverage.loc[
    (coverage.horizon_minutes == 15) & (coverage.window == "primary")
].copy()
# Calculate excluded origins and divide by scheduled origins.
cov["excluded"] = cov.scheduled_rows - cov.eligible_rows
cov["excluded_percent"] = 100 * cov.excluded / cov.scheduled_rows
# Check six assets and 1,497 excluded origins.
assert len(cov) == 6 and cov.excluded.sum() == 1497
print("Coverage by asset; exclusion reasons may overlap:")
display(cov)

# Map model identifiers to the report's family labels.
families = {"log_ridge_harq": "Linear", "lightgbm_qlike": "Trees"}
# Calculate paired session differences; positive values favor B2.
deltas = pd.DataFrame({
    label: ordered[f"loss__{family}__B1"] - ordered[f"loss__{family}__B2"]
    for family, label in families.items()
})
print("Paired session QLIKE differences:")
# Summarize the distribution, including its tails.
display(deltas.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)
# Count strictly positive differences; ties are excluded from favorable sessions.
display(deltas.gt(0).mean().mul(100).rename("favorable_sessions_percent").to_frame())

# Plot exclusions, paired differences, asset results and chronological blocks.
fig, axes = plt.subplots(2, 2, figsize=(13, 9), layout="constrained")
axes[0, 0].bar(cov.asset, cov["excluded_percent"], color="#0072B2")
axes[0, 0].set(title="Coverage: excluded share by asset",
               ylabel="Excluded origins (%)", xlabel="Asset")
# Count sessions in each histogram bin.
for label, color in [("Linear", "#0072B2"), ("Trees", "#D55E00")]:
    axes[0, 1].hist(deltas[label], bins=35, alpha=0.55, color=color, label=label)
axes[0, 1].axvline(0, color="black", linewidth=1)
axes[0, 1].set(title="Full distribution, including tails",
               xlabel="Session QLIKE B1 − B2", ylabel="Number of sessions")
axes[0, 1].legend()
# Select the RV15 B2/B1 development-window contrast.
r = robustness.loc[
    (robustness.horizon_minutes == 15) & (robustness.window == "primary")
    & (robustness.contrast == "B2_over_B1")
]
# Use the same plot structure for assets and chronological blocks.
for axis, prefix, title in [
    (axes[1, 0], "asset_", "Heterogeneity by asset"),
    (axes[1, 1], "chronological_block_", "Chronological blocks"),
]:
    # Select the subset type from its stored prefix.
    subset = r.loc[r.subset.str.startswith(prefix)]
    # Require one row per subset and model family before pivoting.
    assert not subset.duplicated(["subset", "family"]).any()
    table = subset.pivot(index="subset", columns="family",
                         values="qlike_reduction_percent").rename(columns=families)
    # Remove the prefix from displayed subset labels.
    table.index = table.index.str.replace(prefix, "", regex=False)
    table[["Linear", "Trees"]].plot.bar(
        ax=axis, color=["#0072B2", "#D55E00"], rot=0
    )
    axis.axhline(0, color="black", linewidth=1)
    axis.set(title=title, ylabel="QLIKE reduction (%)", xlabel="Subset")
    # Display each subset's session and origin counts.
    display(subset[["subset", "family", "N_sessions", "N_origins",
                    "qlike_reduction_percent"]])
# Add grid lines for reading the plotted values.
for axis in axes.flat:
    axis.grid(axis="y", alpha=0.2)
plt.show()

# Record completion after all checks and figures finish.

# Calculate both the sign frequency and the size of paired differences.
for family,label in families.items():
    series=deltas[label]
    favorable=100*series.gt(0).mean()
    print(f'{label}: {favorable:.2f}% favorable sessions; mean difference={series.mean():+.6f}; minimum={series.min():+.6f}; maximum={series.max():+.6f}.')
# Calculate exclusions from scheduled and eligible origins.
print(f'Scheduled={int(cov.scheduled_rows.sum()):,}; eligible={int(cov.eligible_rows.sum()):,}; excluded={int(cov.excluded.sum()):,}.')
# Count positive results within each model family and subset type.
for family,label in families.items():
    for prefix,unit in [('asset_','assets'),('chronological_block_','blocks')]:
        part=r[(r.family==family)&r.subset.str.startswith(prefix)]
        winners=part[part.qlike_reduction_percent>0].subset.str.replace(prefix,'',regex=False).tolist()
        print(f'{label}: lower B2 loss in {len(winners)}/{len(part)} {unit}: '+', '.join(winners)+'.')
# Recreate the three consecutive date blocks used by aggregate_v4.py.
size=len(ordered)//3
blocks=[ordered.iloc[:size],ordered.iloc[size:2*size],ordered.iloc[2*size:]]
block_dates=[]
for index,block in enumerate(blocks,1):
    recorded=r[r.subset.eq(f'chronological_block_{index}')]
    assert recorded.N_sessions.eq(len(block)).all()
    block_dates.append({'Block':index,'From':block.session_date.iloc[0],'To':block.session_date.iloc[-1],'Sessions':len(block)})
display(pd.DataFrame(block_dates).style.hide(axis='index'))

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_06_session_differences_assets_and_chronological_blocks')


Coverage by asset; exclusion reasons may overlap:


,horizon_minutes,window,asset,eligible_rows,failed_quality_gate,incomplete_mandatory_predictors,invalid_target,scheduled_rows,excluded,excluded_percent
12,15,primary,AAPL,26898,0,155,7,27055,157,0.580299
13,15,primary,AMZN,26743,0,312,0,27055,312,1.153206
14,15,primary,META,26853,0,201,0,27054,201,0.742959
15,15,primary,MSFT,26729,0,326,0,27055,326,1.204953
16,15,primary,NVDA,26764,0,291,0,27055,291,1.075587
17,15,primary,TSLA,26845,0,203,7,27055,210,0.776197


Paired session QLIKE differences:


,count,mean,std,min,5%,25%,50%,75%,95%,max
Linear,419.0,0.001134,0.007658,-0.062803,-0.008460,-0.001615,0.000730,0.003385,0.011879,0.065177
Trees,419.0,-0.000213,0.018935,-0.344417,-0.010711,-0.002304,0.000594,0.003482,0.009796,0.078698


,favorable_sessions_percent
Linear,59.427208
Trees,56.563246


,subset,family,N_sessions,N_origins,qlike_reduction_percent
1,asset_AAPL,log_ridge_harq,419,26898,0.795936
3,asset_AAPL,lightgbm_qlike,419,26898,-0.889023
5,asset_AMZN,log_ridge_harq,419,26743,0.290272
7,asset_AMZN,lightgbm_qlike,419,26743,-0.037613
9,asset_META,log_ridge_harq,419,26853,0.023693
11,asset_META,lightgbm_qlike,419,26853,0.118458
13,asset_MSFT,log_ridge_harq,419,26729,0.439242
15,asset_MSFT,lightgbm_qlike,419,26729,-0.377918
17,asset_NVDA,log_ridge_harq,419,26764,1.263606
19,asset_NVDA,lightgbm_qlike,419,26764,0.568583


,subset,family,N_sessions,N_origins,qlike_reduction_percent
29,chronological_block_1,log_ridge_harq,139,53528,1.181659
31,chronological_block_1,lightgbm_qlike,139,53528,-0.639484
37,chronological_block_2,log_ridge_harq,139,52674,0.273195
39,chronological_block_2,lightgbm_qlike,139,52674,0.613970
45,chronological_block_3,log_ridge_harq,141,54630,0.425095
47,chronological_block_3,lightgbm_qlike,141,54630,-0.265955


Linear: 59.43% favorable sessions; mean difference=+0.001134; minimum=-0.062803; maximum=+0.065177.
Trees: 56.56% favorable sessions; mean difference=-0.000213; minimum=-0.344417; maximum=+0.078698.
Scheduled=162,329; eligible=160,832; excluded=1,497.
Linear: lower B2 loss in 6/6 assets: AAPL, AMZN, META, MSFT, NVDA, TSLA.
Linear: lower B2 loss in 3/3 blocks: 1, 2, 3.
Trees: lower B2 loss in 2/6 assets: META, NVDA.
Trees: lower B2 loss in 1/3 blocks: 2.


Block,From,To,Sessions
1,2024-10-28,2025-06-18,139
2,2025-06-20,2026-01-07,139
3,2026-01-08,2026-07-31,141


### Reading

Linear B2 improves in all six asset summaries and all three blocks. Trees improves for META and NVDA and in the middle block. A majority of favourable sessions can coexist with an adverse mean when losses on other days are larger. Table 8 and Appendix C (Table 19) give these descriptive contrasts; Figures 7 and 8 show distribution and accumulation.


## Reference forecasts

Verify reference losses and matched contrasts by horizon, reference and sample counts. Each matched difference must equal reference QLIKE minus model QLIKE within 1e-12. Both reference masks are retained.


In [7]:
# @title Reference forecasts
# Read all rows in the public reference tables.
reference_qlike = read_public_csv('artifacts/rp4_robustness_public_v1/reference_qlike.csv')
reference_contrasts = read_public_csv('artifacts/rp4_robustness_public_v1/reference_contrasts.csv')
secondary_metrics = read_public_csv('artifacts/rp4_robustness_public_v1/secondary_metrics.csv')
assert [len(reference_qlike),len(reference_contrasts),len(secondary_metrics)] == [4,24,72]
assert set(reference_contrasts.role) == {'POST_PRIMARY_NOMINAL_ROBUSTNESS'}
assert set(secondary_metrics.role) == {'DESCRIPTIVE_NO_TEST'}
# Match each contrast to its reference horizon and sample.
matched = reference_contrasts.merge(reference_qlike[['horizon','reference','N_origins','N_sessions','estimate']], on=['horizon','reference'], suffixes=('','_reference'), validate='many_to_one')
assert (matched.N_sessions == matched.N_sessions_reference).all()
assert (matched.N_origins == matched.N_origins_reference).all()
assert ((matched.reference_qlike - matched.estimate_reference).abs() < 1e-12).all()
assert ((reference_contrasts.reference_qlike-reference_contrasts.model_qlike-reference_contrasts.estimate).abs() < 1e-12).all()
# Display all reference levels and their coverage.
display(reference_qlike[['horizon','reference','N_origins','N_excluded','N_sessions','estimate','ci_low','ci_high']].style.format({'estimate':'{:.6f}','ci_low':'{:.6f}','ci_high':'{:.6f}'}))
# Use each reference's matched sample; lower bars mean lower QLIKE.
fig, axes = plt.subplots(1,2,figsize=(13,4.8),layout='constrained')
for ax, reference in zip(axes, ['seasonal_persistence','HAR_dwm']):
    part=reference_contrasts[(reference_contrasts.horizon==15)&(reference_contrasts.reference==reference)]
    assert len(part)==6
    labels=['Reference']+[families[r.family]+' '+r.set for r in part.itertuples()]
    values=[part.reference_qlike.iloc[0]]+part.model_qlike.tolist()
    ax.bar(labels,values,color=['#777777']+['#0072B2' if f=='log_ridge_harq' else '#D55E00' for f in part.family])
    ax.set(title=f'{reference}: {int(part.N_sessions.iloc[0])} sessions',ylabel='Mean QLIKE (lower is better)')
    ax.tick_params(axis='x',rotation=55)
    ax.grid(axis='y',alpha=.2)
plt.show()
# Display all 24 reference contrasts, grouped by horizon.
for horizon in sorted(reference_contrasts.horizon.unique()):
    print(f'Reference contrasts, RV{horizon}; difference = reference − model:')
    cols=['reference','family','set','N_sessions','reference_qlike','model_qlike','estimate','ci_low','ci_high','p_raw']
    display(reference_contrasts.loc[reference_contrasts.horizon==horizon,cols].style.format({c:'{:.6f}' for c in ['reference_qlike','model_qlike','estimate','ci_low','ci_high','p_raw']}))

# Read the Trees B1 and B2 values on the seasonal-persistence sample.
seasonal_lgb=reference_contrasts[(reference_contrasts.horizon==15)&(reference_contrasts.reference=='seasonal_persistence')&(reference_contrasts.family=='lightgbm_qlike')].set_index('set')

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_07_reference_forecasts')


,horizon,reference,N_origins,N_excluded,N_sessions,estimate,ci_low,ci_high
0,15,seasonal_persistence,159054,1778,417,0.557917,0.513928,0.605794
1,15,HAR_dwm,160832,0,419,0.408061,0.365068,0.477471
2,30,seasonal_persistence,158959,1873,417,0.406483,0.369537,0.446299
3,30,HAR_dwm,160832,0,419,0.339623,0.301285,0.399460


Reference contrasts, RV15; difference = reference − model:


,reference,family,set,N_sessions,reference_qlike,model_qlike,estimate,ci_low,ci_high,p_raw
0,seasonal_persistence,log_ridge_harq,B0,417,0.557917,0.182054,0.375863,0.338188,0.415956,0.000100
1,seasonal_persistence,log_ridge_harq,B1,417,0.557917,0.180591,0.377326,0.339412,0.417769,0.000100
2,seasonal_persistence,log_ridge_harq,B2,417,0.557917,0.179471,0.378446,0.340552,0.418661,0.000100
3,seasonal_persistence,lightgbm_qlike,B0,417,0.557917,0.184349,0.373568,0.336114,0.413360,0.000100
4,seasonal_persistence,lightgbm_qlike,B1,417,0.557917,0.182467,0.375450,0.337845,0.415463,0.000100
5,seasonal_persistence,lightgbm_qlike,B2,417,0.557917,0.181889,0.376028,0.338149,0.416185,0.000100
6,HAR_dwm,log_ridge_harq,B0,419,0.408061,0.183660,0.224400,0.186418,0.283476,0.000100
7,HAR_dwm,log_ridge_harq,B1,419,0.408061,0.182044,0.226017,0.187504,0.286188,0.000100
8,HAR_dwm,log_ridge_harq,B2,419,0.408061,0.180910,0.227150,0.188542,0.287244,0.000100
9,HAR_dwm,lightgbm_qlike,B0,419,0.408061,0.187750,0.220310,0.185181,0.274063,0.000100


Reference contrasts, RV30; difference = reference − model:


,reference,family,set,N_sessions,reference_qlike,model_qlike,estimate,ci_low,ci_high,p_raw
12,seasonal_persistence,log_ridge_harq,B0,417,0.406483,0.146321,0.260162,0.230430,0.292198,0.000100
13,seasonal_persistence,log_ridge_harq,B1,417,0.406483,0.143900,0.262583,0.232566,0.295027,0.000100
14,seasonal_persistence,log_ridge_harq,B2,417,0.406483,0.143093,0.263390,0.233252,0.295677,0.000100
15,seasonal_persistence,lightgbm_qlike,B0,417,0.406483,0.146775,0.259708,0.229752,0.291660,0.000100
16,seasonal_persistence,lightgbm_qlike,B1,417,0.406483,0.143887,0.262596,0.232787,0.294519,0.000100
17,seasonal_persistence,lightgbm_qlike,B2,417,0.406483,0.143522,0.262961,0.233176,0.294791,0.000100
18,HAR_dwm,log_ridge_harq,B0,419,0.339623,0.147307,0.192316,0.160141,0.240612,0.000100
19,HAR_dwm,log_ridge_harq,B1,419,0.339623,0.144760,0.194863,0.161913,0.244792,0.000100
20,HAR_dwm,log_ridge_harq,B2,419,0.339623,0.143957,0.195666,0.162883,0.245438,0.000100
21,HAR_dwm,lightgbm_qlike,B0,419,0.339623,0.149317,0.190306,0.160043,0.234768,0.000100


### Reading

HAR d/w/m has mean QLIKE 0.408061 at 15 minutes. Seasonal persistence uses 417 sessions, whereas HAR uses 419; each comparison belongs to its own mask. Table 11 and Figure 12 provide this context for the nested contrasts in Table 7.


## Complementary errors

Read all 12 RV15 development combinations of family, information set and error metric. MAE is mean absolute error in realised variance; RMSE is root mean squared error. The charts multiply both by one million, with separate vertical scales.


In [8]:
# @title Complementary errors
# Select all RV15 development-window combinations of family, set and metric.
metric_rows=secondary_metrics[(secondary_metrics.horizon==15)&(secondary_metrics.window=='primary')].copy()
assert len(metric_rows)==12
# Scale variance errors by one million for display.
metric_rows['error × 10⁶']=metric_rows.estimate*1e6
metric_rows['CI low × 10⁶']=metric_rows.ci_low*1e6
metric_rows['CI high × 10⁶']=metric_rows.ci_high*1e6
display(metric_rows[['family','set','metric','N_sessions','error × 10⁶','CI low × 10⁶','CI high × 10⁶']].style.format({c:'{:.4f}' for c in ['error × 10⁶','CI low × 10⁶','CI high × 10⁶']}))
# Plot MAE and RMSE separately because their scales differ.
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for ax,metric in zip(axes,['MAE','RMSE']):
    table=metric_rows[metric_rows.metric==metric].pivot(index='set',columns='family',values='error × 10⁶').rename(columns=families)
    table[['Linear','Trees']].plot.bar(ax=ax,rot=0,color=['#0072B2','#D55E00'])
    ax.set(title=f'RV15: {metric}',ylabel='Variance error × 10⁶ (lower is better)',xlabel='Information set')
    ax.grid(axis='y',alpha=.2)
plt.show()
# Calculate the B2/B1 percentage change for each family and metric.
for family,label in families.items():
    for metric in ['MAE','RMSE']:
        values=metric_rows[(metric_rows.family==family)&(metric_rows.metric==metric)].set_index('set').estimate
        change=100*(values['B1']-values['B2'])/values['B1']
        print(f'{label}, {metric}: B1={values["B1"]:.9g}; B2={values["B2"]:.9g}; B2/B1 descriptive reduction={change:+.3f} %.')

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_08_complementary_errors')


,family,set,metric,N_sessions,error × 10⁶,CI low × 10⁶,CI high × 10⁶
0,log_ridge_harq,B0,MAE,419,4.6812,3.5029,6.5898
1,log_ridge_harq,B0,RMSE,419,33.6418,8.5500,56.7060
2,log_ridge_harq,B1,MAE,419,4.7196,3.5023,6.6947
3,log_ridge_harq,B1,RMSE,419,33.5516,8.5570,56.5531
4,log_ridge_harq,B2,MAE,419,4.6911,3.4978,6.6293
5,log_ridge_harq,B2,RMSE,419,33.5190,8.5172,56.5167
6,lightgbm_qlike,B0,MAE,419,4.7374,3.4727,6.8170
7,lightgbm_qlike,B0,RMSE,419,34.7724,8.8382,58.6048
8,lightgbm_qlike,B1,MAE,419,4.7705,3.4666,6.9217
9,lightgbm_qlike,B1,RMSE,419,34.8570,8.8279,58.7401


Linear, MAE: B1=4.71957487e-06; B2=4.69114355e-06; B2/B1 descriptive reduction=+0.602 %.
Linear, RMSE: B1=3.35516459e-05; B2=3.35190183e-05; B2/B1 descriptive reduction=+0.097 %.
Trees, MAE: B1=4.77047347e-06; B2=4.78031957e-06; B2/B1 descriptive reduction=-0.206 %.
Trees, RMSE: B1=3.48570093e-05; B2=3.48762896e-05; B2/B1 descriptive reduction=-0.055 %.


### Reading

Linear B2 slightly reduces MAE and RMSE relative to B1; Trees B2 raises both. These are descriptive complementary errors, with saved intervals rather than additional primary tests. Table 12 gives the original units; Figure 13 uses the scaled display.


## Complete saved results

Inspect all 72 complementary-error records and all 24 historical contrasts, grouped by horizon. The annex retains both development and final windows.


In [9]:
# @title Complete saved results
# Display all 72 secondary-metric rows by horizon and window.
for (horizon,window),part in secondary_metrics.groupby(['horizon','window'],sort=True):
    print(f'RV{horizon} · {window} · secondary metrics')
    display(part[['family','set','metric','N_sessions','N_origins','estimate','ci_low','ci_high']].style.format({c:'{:.6e}' for c in ['estimate','ci_low','ci_high']}))
# Display all 24 contrasts, retaining unavailable formal p-values.
display(statistics[['horizon_minutes']+inference_columns].style.format({'qlike_reduction_percent':'{:+.3f}','ci_low':'{:+.6f}','ci_high':'{:+.6f}','p_for_decision':'{:.4f}'},na_rep='Not tested'))

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_09_complete_saved_results')


RV5 · confirmation · secondary metrics


,family,set,metric,N_sessions,N_origins,estimate,ci_low,ci_high
60,log_ridge_harq,B0,MAE,25,9750,1.315458e-06,1.128079e-06,1.500251e-06
61,log_ridge_harq,B0,RMSE,25,9750,3.584547e-06,2.223974e-06,5.007624e-06
62,log_ridge_harq,B1,MAE,25,9750,1.300025e-06,1.115971e-06,1.480706e-06
63,log_ridge_harq,B1,RMSE,25,9750,3.582528e-06,2.220123e-06,5.005731e-06
64,log_ridge_harq,B2,MAE,25,9750,1.299643e-06,1.115358e-06,1.480639e-06
65,log_ridge_harq,B2,RMSE,25,9750,3.577589e-06,2.220025e-06,4.998941e-06
66,lightgbm_qlike,B0,MAE,25,9750,1.300555e-06,1.109539e-06,1.489432e-06
67,lightgbm_qlike,B0,RMSE,25,9750,3.614830e-06,2.274121e-06,5.022771e-06
68,lightgbm_qlike,B1,MAE,25,9750,1.285895e-06,1.103258e-06,1.468853e-06
69,lightgbm_qlike,B1,RMSE,25,9750,3.589863e-06,2.219102e-06,5.019016e-06


RV5 · primary · secondary metrics


,family,set,metric,N_sessions,N_origins,estimate,ci_low,ci_high
48,log_ridge_harq,B0,MAE,419,160832,2.143127e-06,1.662303e-06,2.896968e-06
49,log_ridge_harq,B0,RMSE,419,160832,1.439014e-05,4.151498e-06,2.408005e-05
50,log_ridge_harq,B1,MAE,419,160832,2.160833e-06,1.665437e-06,2.941049e-06
51,log_ridge_harq,B1,RMSE,419,160832,1.437549e-05,4.167144e-06,2.404447e-05
52,log_ridge_harq,B2,MAE,419,160832,2.155725e-06,1.665381e-06,2.924235e-06
53,log_ridge_harq,B2,RMSE,419,160832,1.437021e-05,4.153903e-06,2.404190e-05
54,lightgbm_qlike,B0,MAE,419,160832,2.142282e-06,1.639716e-06,2.943514e-06
55,lightgbm_qlike,B0,RMSE,419,160832,1.483915e-05,4.250653e-06,2.484389e-05
56,lightgbm_qlike,B1,MAE,419,160832,2.146768e-06,1.635539e-06,2.961433e-06
57,lightgbm_qlike,B1,RMSE,419,160832,1.482152e-05,4.221184e-06,2.482017e-05


RV15 · confirmation · secondary metrics


...

RV30 · confirmation · secondary metrics


,family,set,metric,N_sessions,N_origins,estimate,ci_low,ci_high
36,log_ridge_harq,B0,MAE,25,9750,4.718513e-06,3.943980e-06,5.496663e-06
37,log_ridge_harq,B0,RMSE,25,9750,1.123062e-05,7.050758e-06,1.558189e-05
38,log_ridge_harq,B1,MAE,25,9750,4.605614e-06,3.869981e-06,5.351337e-06
39,log_ridge_harq,B1,RMSE,25,9750,1.118683e-05,6.970649e-06,1.555170e-05
40,log_ridge_harq,B2,MAE,25,9750,4.592197e-06,3.862494e-06,5.330298e-06
41,log_ridge_harq,B2,RMSE,25,9750,1.116549e-05,6.923413e-06,1.553944e-05
42,lightgbm_qlike,B0,MAE,25,9750,4.632513e-06,3.830720e-06,5.476044e-06
43,lightgbm_qlike,B0,RMSE,25,9750,1.124931e-05,7.010086e-06,1.564926e-05
44,lightgbm_qlike,B1,MAE,25,9750,4.490723e-06,3.732005e-06,5.308700e-06
45,lightgbm_qlike,B1,RMSE,25,9750,1.115346e-05,6.790320e-06,1.563568e-05


RV30 · primary · secondary metrics


,family,set,metric,N_sessions,N_origins,estimate,ci_low,ci_high
24,log_ridge_harq,B0,MAE,419,160832,8.025825e-06,6.007886e-06,1.130716e-05
25,log_ridge_harq,B0,RMSE,419,160832,5.103910e-05,1.443391e-05,8.537393e-05
26,log_ridge_harq,B1,MAE,419,160832,8.091325e-06,5.995317e-06,1.149395e-05
27,log_ridge_harq,B1,RMSE,419,160832,5.083330e-05,1.450807e-05,8.497753e-05
28,log_ridge_harq,B2,MAE,419,160832,8.032736e-06,5.994622e-06,1.134820e-05
29,log_ridge_harq,B2,RMSE,419,160832,5.074916e-05,1.440063e-05,8.488664e-05
30,lightgbm_qlike,B0,MAE,419,160832,8.202780e-06,5.960380e-06,1.185905e-05
31,lightgbm_qlike,B0,RMSE,419,160832,5.312622e-05,1.523917e-05,8.868819e-05
32,lightgbm_qlike,B1,MAE,419,160832,8.343600e-06,5.943021e-06,1.230703e-05
33,lightgbm_qlike,B1,RMSE,419,160832,5.341031e-05,1.497689e-05,8.927314e-05


,horizon_minutes,window,family,contrast,N_sessions,qlike_reduction_percent,ci_low,ci_high,p_for_decision,hypothesis_status
0,30,primary,log_ridge_harq,B1_over_B0,419,+1.729,+0.000557,+0.005541,0.0439,REJECTED
1,30,primary,log_ridge_harq,B2_over_B1,419,+0.554,-0.000153,+0.001774,0.0525,NOT_REJECTED
2,30,primary,lightgbm_qlike,B1_over_B0,419,+2.091,+0.001015,+0.005486,0.0053,REJECTED
3,30,primary,lightgbm_qlike,B2_over_B1,419,-0.159,-0.001648,+0.000832,0.6631,NOT_REJECTED
4,30,confirmation,log_ridge_harq,B1_over_B0,25,+0.501,-0.002827,+0.003948,0.3238,NOT_REJECTED
5,30,confirmation,log_ridge_harq,B2_over_B1,25,+1.861,-0.000744,+0.010223,Not tested,NOT_TESTED
6,30,confirmation,lightgbm_qlike,B1_over_B0,25,+1.666,-0.001685,+0.007422,0.1104,NOT_REJECTED
7,30,confirmation,lightgbm_qlike,B2_over_B1,25,-2.414,-0.010379,+0.000025,Not tested,NOT_TESTED
8,15,primary,log_ridge_harq,B1_over_B0,419,+0.880,+0.000253,+0.003454,0.0390,REJECTED
9,15,primary,log_ridge_harq,B2_over_B1,419,+0.623,+0.000335,+0.001932,0.0032,REJECTED


### Reading

The 15-minute extension and the original 30-minute question share the historical research programme. Their rows are separate estimands, not independent replications. Tables 6, 7, 12 and 13 provide the corresponding report summaries.


## A second implementation

Python's standard library reads the same four files and independently reconstructs dates, differences, cumulative sums, quantiles and favourable-session shares. The original checks compare series within 1e-12 QLIKE and percentages within 1e-10 percentage points.


In [10]:
# @title A second implementation
"""Independent cross-check of the public inputs used in the figures."""
# Use the standard library to check the values used in the figures.
import csv
import hashlib
import io
import json
import math

# Identify the four CSV inputs used in these figures.
PATHS = [
    "artifacts/rp4_v4_b4/primary_statistics.csv",
    "artifacts/rp4_v4_b2_rv15/session_losses.csv",
    "artifacts/rp4_v4_b4/coverage.csv",
    "artifacts/rp4_v4_b4/robustness.csv",
]


# Return the reconstructed values and their combined fingerprint.
def crosscheck_public(load_bytes):
    """Return exact-input hashes and all plotted values without fitting."""
    # Read CSV records and calculate fingerprints of the exact input bytes.
    tables, hashes = {}, {}
    for path in PATHS:
        raw = load_bytes(path)
        hashes[path] = hashlib.sha256(raw).hexdigest()
        tables[path] = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
    # Sort sessions before reconstructing series and cumulative differences.
    sessions = sorted(tables[PATHS[1]], key=lambda row: row["session_date"])
    result = {"hashes": hashes, "rows": {p: len(t) for p, t in tables.items()},
              "dates": [r["session_date"] for r in sessions], "families": {}}
    # Require 419 distinct session dates.
    assert len(sessions) == len(set(result["dates"])) == 419
    # Reconstruct Linear and Trees results on paired B1 and B2 dates.
    for family in ("log_ridge_harq", "lightgbm_qlike"):
        baseline = [float(r[f"loss__{family}__B1"]) for r in sessions]
        expanded = [float(r[f"loss__{family}__B2"]) for r in sessions]
        # Subtract paired losses; positive differences favor B2.
        differences = [b - e for b, e in zip(baseline, expanded)]
        ordered = sorted(differences)
        quantiles = []
        # Use linear interpolation between adjacent order statistics, as in pandas.
        for p in (0, .05, .25, .5, .75, .95, 1):
            position = (len(ordered) - 1) * p
            low, high = math.floor(position), math.ceil(position)
            quantiles.append(ordered[low] + (ordered[high] - ordered[low]) * (position - low))
        # Accumulate differences in session order.
        total, cumulative = 0.0, []
        for value in differences:
            total += value
            cumulative.append(total)
        # Average session differences and normalize by the mean B1 loss.
        delta = math.fsum(differences) / len(differences)
        result["families"][family] = {
            "baseline": baseline, "expanded": expanded, "deltas": differences,
            "cumulative": cumulative, "quantiles": quantiles,
            "delta": delta, "percent": 100 * delta / (math.fsum(baseline) / len(baseline)),
            "favorable_percent": 100 * sum(d > 0 for d in differences) / len(differences),
        }
    # Select the horizon, window and contrast used in the descriptive figures.
    result["coverage"] = [r for r in tables[PATHS[2]] if r["horizon_minutes"] == "15" and r["window"] == "primary"]
    result["subsets"] = [r for r in tables[PATHS[3]] if r["horizon_minutes"] == "15" and r["window"] == "primary" and r["contrast"] == "B2_over_B1"]
    # Serialize with fixed ordering to obtain a deterministic fingerprint.
    digest = hashlib.sha256(json.dumps(result, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    return result, digest


# Run the independent cross-check on the embedded CSV bytes.
crosscheck, digest = crosscheck_public(fetch_bytes)
# Compare with the expected fingerprint for the same inputs and calculation.
expected = CROSSCHECK_EXPECTED_DIGEST
# Require identical fingerprints and date ordering before comparing values.
assert digest == expected, "Unexpected input or calculation fingerprint."
assert ordered.session_date.tolist() == crosscheck["dates"]
errors = []
# Check series, differences, cumulative sums and quantiles against plotted values.
for family, label in families.items():
    reference = crosscheck["families"][family]
    for key, observed in [
        ("baseline", ordered[f"loss__{family}__B1"]),
        ("expanded", ordered[f"loss__{family}__B2"]),
        ("deltas", deltas[label]),
        ("cumulative", deltas[label].cumsum()),
        ("quantiles", deltas[label].quantile([0, .05, .25, .5, .75, .95, 1])),
    ]:
        # Require absolute QLIKE differences no larger than 1e-12.
        error = max(abs(a - b) for a, b in zip(observed, reference[key]))
        assert error <= 1e-12, (family, key, error)
        errors.append({"family": label, "object": key, "max_error": error})
    # Check the favorable-session percentage within 1e-10 percentage points.
    assert abs(deltas[label].gt(0).mean() * 100 - reference["favorable_percent"]) <= 1e-10
# Check the total eligible origins across the six assets.
assert cov.eligible_rows.sum() == sum(int(row["eligible_rows"]) for row in crosscheck["coverage"])
# Display the fingerprint, runtime version and maximum numerical differences.
print("PASS: inputs and calculations agree")
print("Runtime:", sys.version.split()[0])
display(pd.DataFrame(errors))

# Record completion after all checks and displays finish.

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_10_a_second_implementation')


PASS: inputs and calculations agree
Runtime: 3.14.2


,family,object,max_error
0,Linear,baseline,9.714451e-17
1,Linear,expanded,9.714451e-17
2,Linear,deltas,9.714451e-17
3,Linear,cumulative,1.720846e-15
4,Linear,quantiles,5.551115e-17
5,Trees,baseline,9.714451e-17
6,Trees,expanded,2.220446e-16
7,Trees,deltas,2.220446e-16
8,Trees,cumulative,7.077672e-16
9,Trees,quantiles,2.220446e-16


### Reading

Agreement checks the arithmetic behind Table 7, Table 8 and Figures 7 and 8. It does not recreate the model fits that produced the session losses.


## Check the values represented

Match every subset by name and model family, then compare its reduction and each asset's excluded fraction with the second implementation. Missing or duplicate rows stop execution.


In [11]:
# @title Check the values represented
# Check subset percentages and exclusion rates against the independent CSV reading.
subset_errors = []
# Match each row by subset and family identifiers.
for row in crosscheck["subsets"]:
    match = r.loc[(r.subset == row["subset"]) & (r.family == row["family"])]
    # Require exactly one matching row.
    assert len(match) == 1
    # Check absolute differences in percentage points.
    error = abs(float(match.iloc[0].qlike_reduction_percent) - float(row["qlike_reduction_percent"]))
    assert error <= 1e-10
    subset_errors.append(error)
# Recompute each asset's excluded share from scheduled and eligible origins.
for row in crosscheck["coverage"]:
    match = cov.loc[cov.asset == row["asset"]].iloc[0]
    expected_rate = 100 * (int(row["scheduled_rows"]) - int(row["eligible_rows"])) / int(row["scheduled_rows"])
    assert abs(match["excluded_percent"] - expected_rate) <= 1e-10
# Display maximum numerical differences and library versions.
print(json.dumps({"status": "PASS_LIGHTWEIGHT_CROSSCHECK",
    "max_series_quantile_error": max(e["max_error"] for e in errors),
    "max_subset_percent_error": max(subset_errors),
    "python": sys.version.split()[0], "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__}, indent=2))

# Record completion after all checks and displays finish.

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_11_check_the_values_represented')


{
  "status": "PASS_LIGHTWEIGHT_CROSSCHECK",
  "max_series_quantile_error": 1.7208456881689926e-15,
  "max_subset_percent_error": 8.673617379884035e-17,
  "python": "3.14.2",
  "pandas": "3.0.5",
  "matplotlib": "3.11.2"
}


### Reading

The maximum discrepancies must satisfy the same declared tolerances. These checks connect the displayed subset and coverage values to Table 3, Table 8 and Table 19.


## Figure 5: Coverage and known-empty windows

Verify the coverage counts against the JSON census, then calculate eligible and known-empty proportions using their stated denominators.


In [12]:
# @title Figure 5: Coverage and known-empty windows
# Verify the six asset rows and the recorded empty-window census.
cov5 = read_public_csv('artifacts/rp4_v4_b4/coverage.csv')
cov5 = cov5[(cov5.horizon_minutes == 15) & (cov5.window == 'primary')].set_index('asset')
empty5 = pd.DataFrame(summary['empty_window_secondary']['census'])
assert len(cov5) == 6 and cov5.index.is_unique
assert not empty5.duplicated(['asset', 'horizon']).any()
assert (empty5.N_empty_origins + empty5.N_nonempty_origins + empty5.N_unknown_origins == empty5.N_origins).all()
assets5 = list(cov5.index)
coverage_figure5 = cov5[['scheduled_rows', 'eligible_rows']].copy()
coverage_figure5['eligible_percent'] = 100 * cov5.eligible_rows / cov5.scheduled_rows
for horizon in ['5m', '30m']:
    counts = empty5[empty5.horizon == horizon].set_index('asset').reindex(assets5)
    assert counts.N_origins.eq(cov5.eligible_rows).all()
    coverage_figure5['empty_' + horizon + '_percent'] = 100 * counts.N_empty_origins / cov5.eligible_rows
assert coverage_figure5.map(lambda v: math.isfinite(float(v))).all().all()
display(coverage_figure5.rename(columns={'scheduled_rows':'Scheduled origins','eligible_rows':'Eligible origins','eligible_percent':'Eligible (%)','empty_5m_percent':'Empty 5-minute (%)','empty_30m_percent':'Empty 30-minute (%)'}).style.format(precision=2))
# Recreate the two coverage panels using the same denominators as the report.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(assets5, cov5.scheduled_rows, color='#d0d0d0', label='Scheduled origins')
axes[0].bar(assets5, cov5.eligible_rows, color='#1f5fa8', label='Eligible origins')
axes[0].set(ylabel='Forecast origins', title='Eligibility after the quality rules')
axes[0].legend()
x = list(range(6))
axes[1].bar([v-.19 for v in x], coverage_figure5.empty_5m_percent, .38, label='5-minute window')
axes[1].bar([v+.19 for v in x], coverage_figure5.empty_30m_percent, .38, label='30-minute window')
axes[1].set(xticks=x, xticklabels=assets5, ylabel='Share of eligible origins (%)', title='Known-empty option windows')
axes[1].legend()
fig.suptitle('Figure 5 · Coverage by asset')
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_12_figure_5_coverage_and_known_empty_windows')


,Scheduled origins,Eligible origins,Eligible (%),Empty 5-minute (%),Empty 30-minute (%)
asset,,,,,
AAPL,27055,26898,99.42,0.26,0.24
AMZN,27055,26743,98.85,0.25,0.24
META,27054,26853,99.26,0.23,0.22
MSFT,27055,26729,98.80,0.24,0.22
NVDA,27055,26764,98.92,0.29,0.27
TSLA,27055,26845,99.22,0.27,0.24


### Reading

Figure 5 and Table 3 describe 160,832 eligible origins across 419 sessions. The empty-window percentages use eligible origins as their denominator; known-empty windows are distinct from unknown or missing observations.


## Figure 6: Primary contrasts and saved intervals

Verify four development contrasts, then rescale each saved difference interval by its own baseline loss.


In [13]:
# @title Figure 6: Primary contrasts and saved intervals
# Select one saved interval for each development-window contrast.
primary6 = statistics[(statistics.horizon_minutes == 15) & (statistics.window == 'primary')].copy()
assert len(primary6) == 4 and not primary6.duplicated(['family','contrast']).any()
assert primary6[['baseline_loss','estimate','ci_low','ci_high']].map(lambda v: math.isfinite(float(v))).all().all()
assert (primary6.baseline_loss > 0).all()
# Rescale the saved QLIKE intervals by their corresponding baseline means.
for field in ['estimate', 'ci_low', 'ci_high']:
    primary6[field + '_percent'] = 100 * primary6[field] / primary6.baseline_loss
assert (primary6.estimate_percent - primary6.qlike_reduction_percent).abs().max() < 1e-10
labels6 = [families[r.family] + ': ' + r.contrast.replace('_over_', ' vs ') for r in primary6.itertuples()]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.errorbar(primary6.estimate_percent, range(4), xerr=[primary6.estimate_percent-primary6.ci_low_percent, primary6.ci_high_percent-primary6.estimate_percent], fmt='o', capsize=4, color='#1f5fa8')
ax.axvline(0, color='grey', lw=1)
ax.set(yticks=range(4), yticklabels=labels6, xlabel='QLIKE reduction (%) with saved 95% percentile interval', title='Figure 6 · Four development-window contrasts')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_13_figure_6_primary_contrasts_and_saved_intervals')


### Reading

Figure 6 displays the four reductions in Table 7 and their saved percentile intervals. The intervals are rescaled into percentage points using each baseline mean; the formal decisions remain those of the fixed H1 → H2 sequence.


## Figures 7 and 8: Session distributions and cumulative paths

Verify all 419 unique session dates, then calculate four paired-difference distributions and the two cumulative mixed-flow paths.


In [14]:
# @title Figures 7 and 8: Session distributions and cumulative paths
# Verify complete, chronological session losses before any display transformation.
ordered78 = losses.sort_values('session_date').copy()
assert len(ordered78) == 419 and ordered78.session_date.is_unique
session_distribution78 = []
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for i, (family, label) in enumerate(families.items()):
    for j, (base, richer) in enumerate([('B0','B1'), ('B1','B2')]):
        delta = ordered78['loss__'+family+'__'+base] - ordered78['loss__'+family+'__'+richer]
        assert delta.map(math.isfinite).all()
        limit = delta.abs().quantile(.99)
        session_distribution78.append({'family':family, 'contrast':richer+'_over_'+base, 'mean':float(delta.mean()), 'median':float(delta.median()), 'positive_sessions':int((delta > 0).sum())})
        # Clip only the displayed histogram; all summaries retain the full values.
        ax = axes[i,j]
        ax.hist(delta.clip(-limit, limit)*1000, bins=45, color=['#1f5fa8','#d9782d'][i], alpha=.65)
        ax.axvline(0, color='grey', lw=1)
        ax.axvline(delta.mean()*1000, label='Mean', color='black')
        ax.axvline(delta.median()*1000, label='Median', color='black', ls='--')
        ax.set(title=label+': '+richer+' vs '+base, xlabel='Session ΔQLIKE (×10⁻³)', ylabel='Sessions')
        ax.legend()
fig.suptitle('Figure 7 · Session-level paired differences')
fig.tight_layout()
plt.show()
# Accumulate the unchanged B1 minus B2 loss differences in session order.
fig, ax = plt.subplots(figsize=(10, 3.8))
cumulative78 = pd.DataFrame({'session':ordered78.session_date})
for family, label in families.items():
    delta = ordered78['loss__'+family+'__B1'] - ordered78['loss__'+family+'__B2']
    cumulative78[family] = delta.cumsum()
    assert math.isclose(float(delta.sum()), float(cumulative78[family].iloc[-1]), rel_tol=0, abs_tol=1e-10)
    ax.plot(pd.to_datetime(ordered78.session_date), cumulative78[family], label=label)
ax.axhline(0, color='grey', lw=1)
ax.set(title='Figure 8 · Cumulative mixed flow block differences', ylabel='Cumulative ΔQLIKE', xlabel='Session')
ax.legend()
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_14_figures_7_and_8_session_distributions_and_cumulative_paths')


### Reading

Figures 7 and 8 show the distribution and timing behind Table 7. The histograms clip absolute differences at the 99th percentile for display only; means, medians and cumulative paths retain every observed value. A positive cumulative change favours the mixed flow block.


## Table 9: Session segments

Verify the segment CSV against the corresponding JSON rows, then select the first- and last-hour mean contrasts.


In [15]:
# @title Table 9: Session segments
# Select saved mean contrasts and compare the CSV with its recorded summary.
regime9 = read_public_csv('artifacts/rp4_v4_b4/regime_secondary.csv')
regime9 = regime9[(regime9.horizon_minutes == 15) & (regime9.window == 'primary') & (regime9.statistic == 'mean')].copy()
assert not regime9.duplicated(['subset','family','contrast']).any()
summary9 = pd.DataFrame(summary['regime_secondary'])
summary9 = summary9[summary9.statistic == 'mean']
joined9 = regime9.merge(summary9, on=['subset','family','contrast'], validate='one_to_one', suffixes=('_csv','_summary'))
assert len(joined9) == len(summary9)
for field in ['estimate','baseline_loss','percent_reduction_mean','N_origins','N_sessions']:
    a, b = joined9[field+'_csv'], joined9[field+'_summary']
    assert (a.isna() == b.isna()).all()
    assert (a.fillna(0) - b.fillna(0)).abs().max() < 1e-10
# Keep unavailable segments explicit instead of substituting a numerical value.
segment_names9 = ['first_hour','last_hour','high_flow','event','weekly_expiration','third_friday','high_gamma']
segments9 = regime9[regime9.subset.isin(segment_names9)].copy()
assert len(segments9) == 28
segments9['Family'] = segments9.family.map(families)
display(segments9[['subset','Family','contrast','N_sessions','N_origins','percent_reduction_mean','p_raw','p_holm','status']].rename(columns={'subset':'Segment','contrast':'Contrast','N_sessions':'Sessions','N_origins':'Origins','percent_reduction_mean':'QLIKE reduction (%)','p_raw':'Two-sided p','p_holm':'Holm p','status':'Status'}).style.format(precision=4, na_rep='Unavailable'))

# Record that this step completed.
EXECUTED_CELLS.append('step_15_table_9_session_segments')


,Segment,Family,Contrast,Sessions,Origins,QLIKE reduction (%),Two-sided p,Holm p,Status
0,first_hour,Linear,B1_over_B0,419,12451,0.4615,0.7023,1.0000,COMPUTED
1,first_hour,Linear,B2_over_B1,419,12451,0.3043,0.3674,1.0000,COMPUTED
2,first_hour,Trees,B1_over_B0,419,12451,1.0773,0.3344,1.0000,COMPUTED
3,first_hour,Trees,B2_over_B1,419,12451,-2.6972,0.5097,1.0000,COMPUTED
12,high_flow,Linear,B1_over_B0,393,75030,0.9234,0.0528,0.1584,COMPUTED
13,high_flow,Linear,B2_over_B1,393,75030,0.5473,0.0784,0.1584,COMPUTED
14,high_flow,Trees,B1_over_B0,393,75030,1.6016,0.0049,0.0196,COMPUTED
15,high_flow,Trees,B2_over_B1,393,75030,0.1217,0.7011,0.7011,COMPUTED
24,event,Linear,B1_over_B0,56,15047,1.2452,0.2594,1.0000,COMPUTED
25,event,Linear,B2_over_B1,56,15047,0.7963,0.2677,1.0000,COMPUTED


### Reading

Table 9 reports these saved segment results. Session counts describe each segment’s coverage, while overlapping segments share observations. Unavailable membership or inference remains explicitly marked rather than treated as a zero effect.


## Table 10: Volatility regimes

Verify the three named regimes and four contrasts per regime. Read their saved means, intervals and secondary probabilities.


In [16]:
# @title Table 10: Volatility regimes
# Verify the three saved variance regimes and their four mean contrasts.
volatility10 = read_public_csv('artifacts/rp4_v4_b4/volatility_regime_secondary.csv')
volatility10 = volatility10[(volatility10.horizon_minutes == 15) & (volatility10.window == 'primary') & (volatility10.statistic == 'mean')].copy()
assert len(volatility10) == 12 and not volatility10.duplicated(['regime','family','contrast']).any()
assert volatility10[['estimate','baseline_loss','percent_reduction_mean','p_raw','p_holm']].map(lambda v: math.isfinite(float(v))).all().all()
assert (100*volatility10.estimate/volatility10.baseline_loss-volatility10.percent_reduction_mean).abs().max() < 1e-10
assert volatility10.groupby('regime').n_sessions.first().sum() == 419
display(volatility10[['regime','family','contrast','n_sessions','percent_reduction_mean','p_raw','p_holm']].rename(columns={'regime':'Variance regime','family':'Family','contrast':'Contrast','n_sessions':'Sessions','percent_reduction_mean':'QLIKE reduction (%)','p_raw':'Two-sided p','p_holm':'Holm p'}).style.format(precision=4))

# Record that this step completed.
EXECUTED_CELLS.append('step_16_table_10_volatility_regimes')


,Variance regime,Family,Contrast,Sessions,QLIKE reduction (%),Two-sided p,Holm p
0,vol_low,Linear,B1_over_B0,140,0.2779,0.3854,1.0000
3,vol_low,Linear,B2_over_B1,140,0.2488,0.4692,1.0000
6,vol_low,Trees,B1_over_B0,140,0.5977,0.2086,0.8344
9,vol_low,Trees,B2_over_B1,140,0.1984,0.5336,1.0000
12,vol_mid,Linear,B1_over_B0,139,0.5620,0.1335,0.3051
15,vol_mid,Linear,B2_over_B1,139,0.4085,0.1017,0.3051
18,vol_mid,Trees,B1_over_B0,139,1.4102,0.0044,0.0176
21,vol_mid,Trees,B2_over_B1,139,0.5662,0.1979,0.3051
24,vol_high,Linear,B1_over_B0,140,1.6025,0.1398,0.4194
27,vol_high,Linear,B2_over_B1,140,1.0909,0.0111,0.0444


### Reading

Table 10 separates low-, middle- and high-variance sessions. These descriptive subsets use the same fitted evaluation; the saved two-sided and Holm-adjusted probabilities retain their secondary role.


## Table 14: Inference sensitivity

Verify the four bilateral probabilities, then reproduce their Holm adjustment by sorting, multiplying and enforcing monotonicity.


In [17]:
# @title Table 14: Inference sensitivity
# Recompute Holm adjustments within each saved group of four contrasts.
holm14 = statistics[(statistics.horizon_minutes == 15) & (statistics.window == 'primary')].copy()
assert len(holm14) == 4 and not holm14.duplicated(['family','contrast']).any()
assert holm14.p_bilateral.between(0,1).all()
sorted14 = holm14.sort_values('p_bilateral')
running14 = 0.0
for rank, (index, row) in enumerate(sorted14.iterrows()):
    running14 = max(running14, min(1.0, (len(sorted14)-rank)*float(row.p_bilateral)))
    holm14.loc[index,'holm_recomputed'] = running14
assert (holm14.holm_recomputed - holm14.p_holm_bilateral).abs().max() < 1e-10
holm14['Family'] = holm14.family.map(families)
display(holm14[['Family','contrast','p_raw','p_bilateral','p_holm_bilateral','holm_recomputed']].rename(columns={'contrast':'Contrast','p_raw':'One-sided p','p_bilateral':'Two-sided p','p_holm_bilateral':'Saved Holm p','holm_recomputed':'Recomputed Holm p'}).style.format(precision=4))

# Record that this step completed.
EXECUTED_CELLS.append('step_17_table_14_inference_sensitivity')


,Family,Contrast,One-sided p,Two-sided p,Saved Holm p,Recomputed Holm p
8,Linear,B1_over_B0,0.0390,0.0459,0.0918,0.0918
9,Linear,B2_over_B1,0.0032,0.0062,0.0248,0.0248
10,Trees,B1_over_B0,0.0135,0.0149,0.0447,0.0447
11,Trees,B2_over_B1,0.6280,0.8048,0.8048,0.8048


### Reading

Table 14 compares the fixed-sequence probabilities with bilateral and Holm-adjusted sensitivity results. The arithmetic above reproduces the saved four-comparison adjustment; it does not change the primary testing sequence.


## Table 17: Availability cutoffs

Verify the two cutoff files and their common 120-second rows before combining the 60-, 120- and 300-second results.


In [18]:
# @title Table 17: Availability cutoffs
# Verify duplicate 120-second rows before combining the two saved files.
cutoff60 = read_public_csv('artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv')
cutoff300 = read_public_csv('artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv')
key17 = ['cutoff_seconds','horizon','family','contrast']
assert cutoff60.columns.equals(cutoff300.columns)
assert cutoff60[cutoff60.cutoff_seconds == 120].sort_values(key17).reset_index(drop=True).equals(cutoff300[cutoff300.cutoff_seconds == 120].sort_values(key17).reset_index(drop=True))
common17 = cutoff60.merge(cutoff300, on=key17, suffixes=('_60','_300'), validate='one_to_one')
assert len(common17) == 2 and common17.cutoff_seconds.eq(120).all()
for field in ['estimate','ci_low','ci_high','p_raw','N_sessions','N_origins']:
    assert (common17[field+'_60'] - common17[field+'_300']).abs().max() < 1e-10
cutoffs17 = pd.concat([cutoff60,cutoff300], ignore_index=True).drop_duplicates(key17).sort_values(['contrast','cutoff_seconds'])
assert len(cutoffs17) == 6 and set(cutoffs17.cutoff_seconds) == {60,120,300}
assert cutoffs17[['estimate','ci_low','ci_high','p_raw']].map(lambda v: math.isfinite(float(v))).all().all()
display(cutoffs17[['cutoff_seconds','contrast','estimate','ci_low','ci_high','p_raw','N_sessions']].rename(columns={'cutoff_seconds':'Cutoff (seconds)','contrast':'Contrast','estimate':'ΔQLIKE','ci_low':'95% interval lower','ci_high':'95% interval upper','p_raw':'One-sided p','N_sessions':'Sessions'}).style.format(precision=6))
fig, axes = plt.subplots(1,2,figsize=(10,3.8))
for ax, contrast in zip(axes,['B1_over_B0','B2_over_B1']):
    rows = cutoffs17[cutoffs17.contrast == contrast]
    ax.errorbar(rows.cutoff_seconds, rows.estimate*1000, yerr=[(rows.estimate-rows.ci_low)*1000,(rows.ci_high-rows.estimate)*1000], fmt='o-', capsize=4)
    ax.axhline(0, color='grey', lw=1)
    ax.set(title=contrast.replace('_over_',' vs '), xticks=[60,120,300], xlabel='Assumed availability cutoff (seconds)', ylabel='ΔQLIKE (×10⁻³)')
fig.suptitle('Figure 17 · Availability-cutoff sensitivity')
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_18_table_17_availability_cutoffs')


,Cutoff (seconds),Contrast,ΔQLIKE,95% interval lower,95% interval upper,One-sided p,Sessions
1,60,B1_over_B0,0.001545,0.000193,0.003384,0.046000,419
0,120,B1_over_B0,0.001616,0.000253,0.003454,0.039000,419
5,300,B1_over_B0,0.001675,0.000372,0.003510,0.034200,419
3,60,B2_over_B1,0.002635,0.001710,0.003582,0.000100,419
2,120,B2_over_B1,0.001134,0.000335,0.001932,0.003200,419
7,300,B2_over_B1,0.000353,-0.000445,0.001150,0.194800,419


### Reading

Table 17 and Figure 17 compare assumed 60-, 120- and 300-second availability cutoffs for the linear family. Their intervals and probabilities are saved sensitivity results; the 120-second rows appear once after the duplicate check.


## Figure 16: Shuffled mixed-flow comparisons

Verify 50 saved permutations at each horizon, then recompute their means, exceedance counts and empirical probabilities.


In [19]:
# @title Figure 16: Shuffled mixed-flow comparisons
# Read the 50 saved shuffled-flow differences for each target horizon.
placebo_rows15 = []
fig, axes = plt.subplots(1,2,figsize=(10,3.8))
for ax, horizon in zip(axes,[15,30]):
    draws = read_public_csv('artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv'+str(horizon)+'_draws.csv')
    assert len(draws) == 50 and draws.k.is_unique and draws.delta.map(math.isfinite).all()
    observed = statistics[(statistics.horizon_minutes == horizon) & (statistics.window == 'primary') & (statistics.family == 'log_ridge_harq') & (statistics.contrast == 'B2_over_B1')]
    assert len(observed) == 1
    observed_delta = float(observed.estimate.iloc[0])
    exceedances = int((draws.delta >= observed_delta).sum())
    empirical_p = (exceedances+1)/(len(draws)+1)
    placebo_rows15.append({'horizon':horizon,'observed_delta':observed_delta,'placebo_mean':float(draws.delta.mean()),'exceedances':exceedances,'draws':len(draws),'p_empirical':empirical_p})
    # Plot saved differences; count exceedances on their unrounded values.
    ax.hist(draws.delta*1000, bins=12, color='grey', alpha=.55, label='Saved shuffled-flow differences')
    ax.axvline(observed_delta*1000, color='#1f5fa8', lw=2, label='Aligned mixed flow block')
    ax.axvline(draws.delta.mean()*1000, color='black', ls='--', label='Shuffled mean')
    ax.set(title='RV'+str(horizon)+' · Linear', xlabel='ΔQLIKE (×10⁻³)', ylabel='Permutations')
    ax.legend(fontsize=8)
placebo15 = pd.DataFrame(placebo_rows15)
display(placebo15.rename(columns={'horizon':'Horizon (minutes)','observed_delta':'Observed ΔQLIKE','placebo_mean':'Shuffled mean ΔQLIKE','exceedances':'Exceedances','draws':'Permutations','p_empirical':'Empirical p'}).style.format(precision=6))
fig.suptitle('Figure 16 · Aligned and shuffled mixed flow block')
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_19_figure_15_shuffled_mixed_flow_comparisons')


,Horizon (minutes),Observed ΔQLIKE,Shuffled mean ΔQLIKE,Exceedances,Permutations,Empirical p
0,15,0.001134,0.000926,12,50,0.254902
1,30,0.000802,0.001211,38,50,0.764706


### Reading

Figure 16 and Section 4.10 compare the aligned mixed flow block with 50 saved shuffled versions at each horizon. The empirical probability uses (exceedances + 1) / 51, so its resolution is limited by the number of saved permutations.


## Figure 11: Subset comparisons

Match every asset, chronological block and variance tercile to one result per family and contrast, then place the percentages on a shared scale.


In [20]:
# @title Figure 11: Subset comparisons
# Assemble the saved asset, chronological-block and variance-regime reductions.
robust11 = read_public_csv('artifacts/rp4_v4_b4/robustness.csv')
robust11 = robust11[(robust11.horizon_minutes == 15) & (robust11.window == 'primary')]
row_keys11 = ['asset_'+a for a in ['AAPL','AMZN','META','MSFT','NVDA','TSLA']] + ['chronological_block_'+str(i) for i in [1,2,3]] + ['vol_low','vol_mid','vol_high']
column_keys11 = [(f,c) for f in families for c in ['B1_over_B0','B2_over_B1']]
heatmap11 = pd.DataFrame(index=row_keys11, columns=[families[f]+': '+c.replace('_over_',' vs ') for f,c in column_keys11], dtype=float)
for subset in row_keys11:
    for family, contrast in column_keys11:
        if subset.startswith('vol_'):
            value = volatility10[(volatility10.regime == subset) & (volatility10.family == families[family]) & (volatility10.contrast == contrast)].percent_reduction_mean
        else:
            value = robust11[(robust11.subset == subset) & (robust11.family == family) & (robust11.contrast == contrast)].qlike_reduction_percent
        assert len(value) == 1 and math.isfinite(float(value.iloc[0]))
        heatmap11.loc[subset, families[family]+': '+contrast.replace('_over_',' vs ')] = float(value.iloc[0])
# Use a symmetric colour scale so equally sized gains and losses have equal weight.
fig, ax = plt.subplots(figsize=(8,6))
limit11 = float(heatmap11.abs().max().max())
im = ax.imshow(heatmap11.to_numpy(), cmap='RdBu', vmin=-limit11, vmax=limit11, aspect='auto')
ax.set(xticks=range(4), xticklabels=[c.replace(': ', chr(10)) for c in heatmap11.columns], yticks=range(12), yticklabels=[r.replace('asset_','').replace('chronological_block_','Block ').replace('vol_','Variance: ') for r in row_keys11], title='Figure 11 · Reductions across descriptive subsets')
for r in range(12):
    for c in range(4):
        v = float(heatmap11.iloc[r,c])
        ax.text(c,r,f'{v:+.2f}',ha='center',va='center',color='white' if abs(v)>.6*limit11 else 'black')
fig.colorbar(im,ax=ax,label='QLIKE reduction (%)')
fig.tight_layout()
plt.show()

# Record that this step completed.
EXECUTED_CELLS.append('step_20_figure_11_subset_comparisons')


### Reading

Figure 11 places the asset, chronological-block and variance-regime results on one percentage scale. The entries reproduce Tables 8, 10 and 19; colour differences describe subsets of one evaluation rather than separate primary tests.


## Claims ledger

Each report value below is transcribed from the PDF. The comparison column distinguishes paired-loss calculations from saved statistics read from verified files. Tolerances are absolute: 0.0005 percentage points for three-decimal reductions, 1e-6 QLIKE, and exact equality for counts and decision states. Probabilities use 0.00005 to match four-decimal printing. Other rounded percentages use half their printed unit, shown on each row. Complementary errors use their printed precision in realised-variance units. Any mismatch stops the notebook.

The checks cover Tables 3, 6 to 14, 17 and 19, plus the placebo values in Section 4.10. Table 17's favourable-session counts are recalculated from paired losses at all three cutoffs.


In [21]:
# @title Claims ledger
# Fixed values transcribed from the report; never replace them with source-file values.
REPORT_CLAIMS = [{'report_location': 'Table 6', 'claim': 'Linear H1: estimate', 'report_literal': '+0.002547', 'report_value': 0.002547, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H1: ci_low', 'report_literal': '+0.000557', 'report_value': 0.000557, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H1: ci_high', 'report_literal': '+0.005541', 'report_value': 0.005541, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H1: percent_reduction_mean', 'report_literal': '+1.729', 'report_value': 1.729, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H1: p_raw', 'report_literal': '0.0439', 'report_value': 0.0439, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 6', 'claim': 'Linear H2: estimate', 'report_literal': '+0.000802', 'report_value': 0.000802, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H2: ci_low', 'report_literal': '-0.000153', 'report_value': -0.000153, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H2: ci_high', 'report_literal': '+0.001774', 'report_value': 0.001774, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H2: percent_reduction_mean', 'report_literal': '+0.554', 'report_value': 0.554, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H2: p_raw', 'report_literal': '0.0525', 'report_value': 0.0525, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Linear H2: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 6', 'claim': 'Trees H1: estimate', 'report_literal': '+0.003122', 'report_value': 0.003122, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H1: ci_low', 'report_literal': '+0.001015', 'report_value': 0.001015, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H1: ci_high', 'report_literal': '+0.005486', 'report_value': 0.005486, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H1: percent_reduction_mean', 'report_literal': '+2.091', 'report_value': 2.091, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0053', 'report_value': 0.0053, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 6', 'claim': 'Trees H2: estimate', 'report_literal': '-0.000233', 'report_value': -0.000233, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H2: ci_low', 'report_literal': '-0.001648', 'report_value': -0.001648, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H2: ci_high', 'report_literal': '+0.000832', 'report_value': 0.000832, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H2: percent_reduction_mean', 'report_literal': '-0.159', 'report_value': -0.159, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H2: p_raw', 'report_literal': '0.6631', 'report_value': 0.6631, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 6', 'claim': 'Trees H2: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 30, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Linear H1: estimate', 'report_literal': '+0.001616', 'report_value': 0.001616, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: ci_low', 'report_literal': '+0.000253', 'report_value': 0.000253, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: ci_high', 'report_literal': '+0.003454', 'report_value': 0.003454, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: percent_reduction_mean', 'report_literal': '+0.880', 'report_value': 0.88, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: p_raw', 'report_literal': '0.0390', 'report_value': 0.039, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Linear H2: estimate', 'report_literal': '+0.001134', 'report_value': 0.001134, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: ci_low', 'report_literal': '+0.000335', 'report_value': 0.000335, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: ci_high', 'report_literal': '+0.001932', 'report_value': 0.001932, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: percent_reduction_mean', 'report_literal': '+0.623', 'report_value': 0.623, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: p_raw', 'report_literal': '0.0032', 'report_value': 0.0032, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Trees H1: estimate', 'report_literal': '+0.002197', 'report_value': 0.002197, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: ci_low', 'report_literal': '+0.000611', 'report_value': 0.000611, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: ci_high', 'report_literal': '+0.004103', 'report_value': 0.004103, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: percent_reduction_mean', 'report_literal': '+1.170', 'report_value': 1.17, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0135', 'report_value': 0.0135, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Trees H2: estimate', 'report_literal': '-0.000213', 'report_value': -0.000213, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: ci_low', 'report_literal': '-0.002044', 'report_value': -0.002044, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: ci_high', 'report_literal': '+0.001104', 'report_value': 0.001104, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: percent_reduction_mean', 'report_literal': '-0.115', 'report_value': -0.115, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: p_raw', 'report_literal': '0.6280', 'report_value': 0.628, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Linear H1: estimate', 'report_literal': '+0.000416', 'report_value': 0.000416, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: ci_low', 'report_literal': '-0.002209', 'report_value': -0.002209, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: ci_high', 'report_literal': '+0.002878', 'report_value': 0.002878, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: percent_reduction_mean', 'report_literal': '+0.174', 'report_value': 0.174, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: p_raw', 'report_literal': '0.3908', 'report_value': 0.3908, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Linear H2: estimate', 'report_literal': '+0.004767', 'report_value': 0.004767, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: ci_low', 'report_literal': '-0.001109', 'report_value': -0.001109, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: ci_high', 'report_literal': '+0.014742', 'report_value': 0.014742, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: percent_reduction_mean', 'report_literal': '+1.997', 'report_value': 1.997, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: p_raw', 'report_literal': '0.1758', 'report_value': 0.1758, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: decision', 'report_literal': 'Gate closed', 'report_value': 'Gate closed', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Trees H1: estimate', 'report_literal': '+0.007482', 'report_value': 0.007482, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: ci_low', 'report_literal': '+0.001644', 'report_value': 0.001644, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: ci_high', 'report_literal': '+0.016807', 'report_value': 0.016807, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: percent_reduction_mean', 'report_literal': '+3.163', 'report_value': 3.163, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0568', 'report_value': 0.0568, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Trees H2: estimate', 'report_literal': '-0.002272', 'report_value': -0.002272, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: ci_low', 'report_literal': '-0.009072', 'report_value': -0.009072, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: ci_high', 'report_literal': '+0.002195', 'report_value': 0.002195, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: percent_reduction_mean', 'report_literal': '-0.992', 'report_value': -0.992, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: p_raw', 'report_literal': '0.7554', 'report_value': 0.7554, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: decision', 'report_literal': 'Gate closed', 'report_value': 'Gate closed', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 8', 'claim': 'AAPL Linear: reduction', 'report_literal': '+0.796', 'report_value': 0.796, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AAPL Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AAPL Trees: reduction', 'report_literal': '-0.889', 'report_value': -0.889, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AAPL Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AMZN Linear: reduction', 'report_literal': '+0.290', 'report_value': 0.29, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AMZN Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AMZN Trees: reduction', 'report_literal': '-0.038', 'report_value': -0.038, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'AMZN Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'META Linear: reduction', 'report_literal': '+0.024', 'report_value': 0.024, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'META Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'META Trees: reduction', 'report_literal': '+0.118', 'report_value': 0.118, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'META Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'MSFT Linear: reduction', 'report_literal': '+0.439', 'report_value': 0.439, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'MSFT Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'MSFT Trees: reduction', 'report_literal': '-0.378', 'report_value': -0.378, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'MSFT Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'NVDA Linear: reduction', 'report_literal': '+1.264', 'report_value': 1.264, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'NVDA Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'NVDA Trees: reduction', 'report_literal': '+0.569', 'report_value': 0.569, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'NVDA Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'TSLA Linear: reduction', 'report_literal': '+0.952', 'report_value': 0.952, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'TSLA Linear: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'TSLA Trees: reduction', 'report_literal': '-0.024', 'report_value': -0.024, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'TSLA Trees: sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 1 Linear: reduction', 'report_literal': '+1.182', 'report_value': 1.182, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_1'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 1 Linear: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_1'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 1 Trees: reduction', 'report_literal': '-0.639', 'report_value': -0.639, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_1'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 1 Trees: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_1'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 2 Linear: reduction', 'report_literal': '+0.273', 'report_value': 0.273, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_2'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 2 Linear: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_2'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 2 Trees: reduction', 'report_literal': '+0.614', 'report_value': 0.614, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_2'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 2 Trees: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_2'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 3 Linear: reduction', 'report_literal': '+0.425', 'report_value': 0.425, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_3'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 3 Linear: sessions', 'report_literal': '141', 'report_value': 141.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_3'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 3 Trees: reduction', 'report_literal': '-0.266', 'report_value': -0.266, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_3'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 8', 'claim': 'Block 3 Trees: sessions', 'report_literal': '141', 'report_value': 141.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'chronological_block_3'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: estimate', 'report_literal': '+0.000649', 'report_value': 0.000649, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: percent_reduction_mean', 'report_literal': '+0.461', 'report_value': 0.461, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: ci_low', 'report_literal': '-0.001796', 'report_value': -0.001796, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: ci_high', 'report_literal': '+0.004548', 'report_value': 0.004548, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: p_raw', 'report_literal': '0.7023', 'report_value': 0.7023, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B1/B0: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: estimate', 'report_literal': '+0.000426', 'report_value': 0.000426, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: percent_reduction_mean', 'report_literal': '+0.304', 'report_value': 0.304, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: ci_low', 'report_literal': '-0.000502', 'report_value': -0.000502, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: ci_high', 'report_literal': '+0.001366', 'report_value': 0.001366, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: p_raw', 'report_literal': '0.3674', 'report_value': 0.3674, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Linear B2/B1: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: estimate', 'report_literal': '+0.001686', 'report_value': 0.001686, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: percent_reduction_mean', 'report_literal': '+1.077', 'report_value': 1.077, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: ci_low', 'report_literal': '-0.001230', 'report_value': -0.00123, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: ci_high', 'report_literal': '+0.005691', 'report_value': 0.005691, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: p_raw', 'report_literal': '0.3344', 'report_value': 0.3344, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B1/B0: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: estimate', 'report_literal': '-0.004176', 'report_value': -0.004176, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: percent_reduction_mean', 'report_literal': '-2.697', 'report_value': -2.697, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: ci_low', 'report_literal': '-0.013971', 'report_value': -0.013971, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: ci_high', 'report_literal': '+0.001206', 'report_value': 0.001206, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: p_raw', 'report_literal': '0.5097', 'report_value': 0.5097, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'First hour Trees B2/B1: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'first_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: estimate', 'report_literal': '+0.001019', 'report_value': 0.001019, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: percent_reduction_mean', 'report_literal': '+0.508', 'report_value': 0.508, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: ci_low', 'report_literal': '-0.000892', 'report_value': -0.000892, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: ci_high', 'report_literal': '+0.003120', 'report_value': 0.00312, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: p_raw', 'report_literal': '0.3236', 'report_value': 0.3236, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B1/B0: p_holm', 'report_literal': '0.6472', 'report_value': 0.6472, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: estimate', 'report_literal': '+0.001442', 'report_value': 0.001442, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: percent_reduction_mean', 'report_literal': '+0.723', 'report_value': 0.723, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: ci_low', 'report_literal': '+0.000129', 'report_value': 0.000129, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: ci_high', 'report_literal': '+0.002815', 'report_value': 0.002815, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: p_raw', 'report_literal': '0.0364', 'report_value': 0.0364, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Linear B2/B1: p_holm', 'report_literal': '0.1092', 'report_value': 0.1092, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: estimate', 'report_literal': '+0.003171', 'report_value': 0.003171, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: percent_reduction_mean', 'report_literal': '+1.575', 'report_value': 1.575, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: ci_low', 'report_literal': '+0.000601', 'report_value': 0.000601, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: ci_high', 'report_literal': '+0.005928', 'report_value': 0.005928, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: p_raw', 'report_literal': '0.0190', 'report_value': 0.019, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B1/B0: p_holm', 'report_literal': '0.0760', 'report_value': 0.076, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: estimate', 'report_literal': '+0.000196', 'report_value': 0.000196, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: percent_reduction_mean', 'report_literal': '+0.099', 'report_value': 0.099, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: ci_low', 'report_literal': '-0.001484', 'report_value': -0.001484, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: ci_high', 'report_literal': '+0.001902', 'report_value': 0.001902, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: p_raw', 'report_literal': '0.8179', 'report_value': 0.8179, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 9', 'claim': 'Last hour Trees B2/B1: p_holm', 'report_literal': '0.8179', 'report_value': 0.8179, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#regime_secondary', 'filter': {'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'subset': 'last_hour'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: estimate', 'report_literal': '+0.000456', 'report_value': 0.000456, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: percent_reduction_mean', 'report_literal': '+0.278', 'report_value': 0.278, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: ci_low', 'report_literal': '-0.000594', 'report_value': -0.000594, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: ci_high', 'report_literal': '+0.001459', 'report_value': 0.001459, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: p_raw', 'report_literal': '0.3854', 'report_value': 0.3854, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B1/B0: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: estimate', 'report_literal': '+0.000407', 'report_value': 0.000407, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: percent_reduction_mean', 'report_literal': '+0.249', 'report_value': 0.249, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: ci_low', 'report_literal': '-0.000781', 'report_value': -0.000781, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: ci_high', 'report_literal': '+0.001417', 'report_value': 0.001417, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: p_raw', 'report_literal': '0.4692', 'report_value': 0.4692, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Linear B2/B1: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: estimate', 'report_literal': '+0.000994', 'report_value': 0.000994, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: percent_reduction_mean', 'report_literal': '+0.598', 'report_value': 0.598, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: ci_low', 'report_literal': '-0.000526', 'report_value': -0.000526, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: ci_high', 'report_literal': '+0.002555', 'report_value': 0.002555, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: p_raw', 'report_literal': '0.2086', 'report_value': 0.2086, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: p_holm', 'report_literal': '0.8344', 'report_value': 0.8344, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B1/B0: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: estimate', 'report_literal': '+0.000328', 'report_value': 0.000328, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: percent_reduction_mean', 'report_literal': '+0.198', 'report_value': 0.198, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: ci_low', 'report_literal': '-0.000729', 'report_value': -0.000729, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: ci_high', 'report_literal': '+0.001363', 'report_value': 0.001363, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: p_raw', 'report_literal': '0.5336', 'report_value': 0.5336, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: p_holm', 'report_literal': '1.0000', 'report_value': 1.0, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Low Trees B2/B1: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_low'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: estimate', 'report_literal': '+0.000977', 'report_value': 0.000977, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: percent_reduction_mean', 'report_literal': '+0.562', 'report_value': 0.562, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: ci_low', 'report_literal': '-0.000310', 'report_value': -0.00031, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: ci_high', 'report_literal': '+0.002251', 'report_value': 0.002251, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: p_raw', 'report_literal': '0.1335', 'report_value': 0.1335, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: p_holm', 'report_literal': '0.3051', 'report_value': 0.3051, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B1/B0: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: estimate', 'report_literal': '+0.000706', 'report_value': 0.000706, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: percent_reduction_mean', 'report_literal': '+0.408', 'report_value': 0.408, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: ci_low', 'report_literal': '-0.000116', 'report_value': -0.000116, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: ci_high', 'report_literal': '+0.001563', 'report_value': 0.001563, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: p_raw', 'report_literal': '0.1017', 'report_value': 0.1017, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: p_holm', 'report_literal': '0.3051', 'report_value': 0.3051, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Linear B2/B1: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: estimate', 'report_literal': '+0.002455', 'report_value': 0.002455, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: percent_reduction_mean', 'report_literal': '+1.410', 'report_value': 1.41, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: ci_low', 'report_literal': '+0.000840', 'report_value': 0.00084, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: ci_high', 'report_literal': '+0.004166', 'report_value': 0.004166, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: p_raw', 'report_literal': '0.0044', 'report_value': 0.0044, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: p_holm', 'report_literal': '0.0176', 'report_value': 0.0176, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B1/B0: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: estimate', 'report_literal': '+0.000972', 'report_value': 0.000972, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: percent_reduction_mean', 'report_literal': '+0.566', 'report_value': 0.566, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: ci_low', 'report_literal': '-0.000501', 'report_value': -0.000501, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: ci_high', 'report_literal': '+0.002454', 'report_value': 0.002454, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: p_raw', 'report_literal': '0.1979', 'report_value': 0.1979, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: p_holm', 'report_literal': '0.3051', 'report_value': 0.3051, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'Mid Trees B2/B1: sessions', 'report_literal': '139', 'report_value': 139.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_mid'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: estimate', 'report_literal': '+0.003411', 'report_value': 0.003411, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: percent_reduction_mean', 'report_literal': '+1.603', 'report_value': 1.603, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: ci_low', 'report_literal': '-0.000485', 'report_value': -0.000485, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: ci_high', 'report_literal': '+0.008728', 'report_value': 0.008728, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: p_raw', 'report_literal': '0.1398', 'report_value': 0.1398, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: p_holm', 'report_literal': '0.4194', 'report_value': 0.4194, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B1/B0: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: estimate', 'report_literal': '+0.002285', 'report_value': 0.002285, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: percent_reduction_mean', 'report_literal': '+1.091', 'report_value': 1.091, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: ci_low', 'report_literal': '+0.000551', 'report_value': 0.000551, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: ci_high', 'report_literal': '+0.004088', 'report_value': 0.004088, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: p_raw', 'report_literal': '0.0111', 'report_value': 0.0111, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: p_holm', 'report_literal': '0.0444', 'report_value': 0.0444, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Linear B2/B1: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Linear', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: estimate', 'report_literal': '+0.003143', 'report_value': 0.003143, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: percent_reduction_mean', 'report_literal': '+1.411', 'report_value': 1.411, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: ci_low', 'report_literal': '-0.000960', 'report_value': -0.00096, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: ci_high', 'report_literal': '+0.008516', 'report_value': 0.008516, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: p_raw', 'report_literal': '0.1806', 'report_value': 0.1806, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: p_holm', 'report_literal': '0.4194', 'report_value': 0.4194, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B1/B0: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B1_over_B0', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: estimate', 'report_literal': '-0.001930', 'report_value': -0.00193, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: percent_reduction_mean', 'report_literal': '-0.879', 'report_value': -0.879, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: ci_low', 'report_literal': '-0.007090', 'report_value': -0.00709, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: ci_high', 'report_literal': '+0.001508', 'report_value': 0.001508, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: p_raw', 'report_literal': '0.3936', 'report_value': 0.3936, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: p_holm', 'report_literal': '0.4194', 'report_value': 0.4194, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'p_holm', 'operation': None}, {'report_location': 'Table 10', 'claim': 'High Trees B2/B1: sessions', 'report_literal': '140', 'report_value': 140.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv', 'filter': {'family': 'Trees', 'contrast': 'B2_over_B1', 'statistic': 'mean', 'regime': 'vol_high'}, 'column': 'n_sessions', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: N_sessions', 'report_literal': '417', 'report_value': 417.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: N_origins', 'report_literal': '159,054', 'report_value': 159054.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: reference_qlike', 'report_literal': '0.557917', 'report_value': 0.557917, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'reference_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: model_qlike', 'report_literal': '0.182054', 'report_value': 0.182054, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'model_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: p_raw', 'report_literal': '0.0001', 'report_value': 0.0001, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: N_sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: N_origins', 'report_literal': '160,832', 'report_value': 160832.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: reference_qlike', 'report_literal': '0.408061', 'report_value': 0.408061, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'reference_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: model_qlike', 'report_literal': '0.183660', 'report_value': 0.18366, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'model_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: p_raw', 'report_literal': '0.0001', 'report_value': 0.0001, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B0: MAE', 'report_literal': '4.68118e-6', 'report_value': 4.68118e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B0', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B0: RMSE', 'report_literal': '3.36418e-5', 'report_value': 3.36418e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B0', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B1: MAE', 'report_literal': '4.71957e-6', 'report_value': 4.71957e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B1', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B1: RMSE', 'report_literal': '3.35516e-5', 'report_value': 3.35516e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B1', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B2: MAE', 'report_literal': '4.69114e-6', 'report_value': 4.69114e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B2', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B2: RMSE', 'report_literal': '3.35190e-5', 'report_value': 3.3519e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B2', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B0: MAE', 'report_literal': '4.73740e-6', 'report_value': 4.7374e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B0', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B0: RMSE', 'report_literal': '3.47724e-5', 'report_value': 3.47724e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B0', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B1: MAE', 'report_literal': '4.77047e-6', 'report_value': 4.77047e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B1', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B1: RMSE', 'report_literal': '3.48570e-5', 'report_value': 3.4857e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B1', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B2: MAE', 'report_literal': '4.78032e-6', 'report_value': 4.78032e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B2', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B2: RMSE', 'report_literal': '3.48763e-5', 'report_value': 3.48763e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B2', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H1: p_raw', 'report_literal': '0.0390', 'report_value': 0.039, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H1: p_bilateral', 'report_literal': '0.0459', 'report_value': 0.0459, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H1: p_holm_bilateral', 'report_literal': '0.0918', 'report_value': 0.0918, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_holm_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H2: p_raw', 'report_literal': '0.0032', 'report_value': 0.0032, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H2: p_bilateral', 'report_literal': '0.0062', 'report_value': 0.0062, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Linear H2: p_holm_bilateral', 'report_literal': '0.0248', 'report_value': 0.0248, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_holm_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0135', 'report_value': 0.0135, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H1: p_bilateral', 'report_literal': '0.0149', 'report_value': 0.0149, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H1: p_holm_bilateral', 'report_literal': '0.0447', 'report_value': 0.0447, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_holm_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H2: p_raw', 'report_literal': '0.6280', 'report_value': 0.628, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H2: p_bilateral', 'report_literal': '0.8048', 'report_value': 0.8048, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_bilateral', 'operation': None}, {'report_location': 'Table 14', 'claim': 'Trees H2: p_holm_bilateral', 'report_literal': '0.8048', 'report_value': 0.8048, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_holm_bilateral', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s State: estimate', 'report_literal': '.001545', 'report_value': 0.001545, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s State: ci_low', 'report_literal': '.000193', 'report_value': 0.000193, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s State: ci_high', 'report_literal': '.003384', 'report_value': 0.003384, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s State: p_raw', 'report_literal': '.0460', 'report_value': 0.046, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s State: reduction', 'report_literal': '.841', 'report_value': 0.841, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 17', 'claim': '60s Flow: estimate', 'report_literal': '.002635', 'report_value': 0.002635, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s Flow: ci_low', 'report_literal': '.001710', 'report_value': 0.00171, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s Flow: ci_high', 'report_literal': '.003582', 'report_value': 0.003582, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s Flow: p_raw', 'report_literal': '.0001', 'report_value': 0.0001, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '60s Flow: reduction', 'report_literal': '1.447', 'report_value': 1.447, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 60, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 17', 'claim': '120s State: estimate', 'report_literal': '.001616', 'report_value': 0.001616, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s State: ci_low', 'report_literal': '.000253', 'report_value': 0.000253, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s State: ci_high', 'report_literal': '.003454', 'report_value': 0.003454, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s State: p_raw', 'report_literal': '.0390', 'report_value': 0.039, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s State: reduction', 'report_literal': '.880', 'report_value': 0.88, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 17', 'claim': '120s Flow: estimate', 'report_literal': '.001134', 'report_value': 0.001134, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s Flow: ci_low', 'report_literal': '.000335', 'report_value': 0.000335, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s Flow: ci_high', 'report_literal': '.001932', 'report_value': 0.001932, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s Flow: p_raw', 'report_literal': '.0032', 'report_value': 0.0032, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '120s Flow: reduction', 'report_literal': '.623', 'report_value': 0.623, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv', 'filter': {'cutoff_seconds': 120, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 17', 'claim': '300s State: estimate', 'report_literal': '.001675', 'report_value': 0.001675, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s State: ci_low', 'report_literal': '.000372', 'report_value': 0.000372, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s State: ci_high', 'report_literal': '.003510', 'report_value': 0.00351, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s State: p_raw', 'report_literal': '.0342', 'report_value': 0.0342, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s State: reduction', 'report_literal': '.912', 'report_value': 0.912, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 17', 'claim': '300s Flow: estimate', 'report_literal': '.000353', 'report_value': 0.000353, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s Flow: ci_low', 'report_literal': '-.000445', 'report_value': -0.000445, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s Flow: ci_high', 'report_literal': '.001150', 'report_value': 0.00115, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s Flow: p_raw', 'report_literal': '.1948', 'report_value': 0.1948, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 17', 'claim': '300s Flow: reduction', 'report_literal': '.194', 'report_value': 0.194, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv', 'filter': {'cutoff_seconds': 300, 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': '100*estimate/(primary_Ridge_B0_loss - cutoff_state_estimate if Flow else primary_Ridge_B0_loss)'}, {'report_location': 'Table 19', 'claim': 'AAPL R H1: N_origins', 'report_literal': '26898', 'report_value': 26898.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL R H1: estimate', 'report_literal': '+0.0024703', 'report_value': 0.0024703, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL R H1: qlike_reduction_percent', 'report_literal': '+1.2695', 'report_value': 1.2695, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL R H2: N_origins', 'report_literal': '26898', 'report_value': 26898.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL R H2: estimate', 'report_literal': '+0.0015291', 'report_value': 0.0015291, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL R H2: qlike_reduction_percent', 'report_literal': '+0.7959', 'report_value': 0.7959, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H1: N_origins', 'report_literal': '26898', 'report_value': 26898.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H1: estimate', 'report_literal': '+0.0018376', 'report_value': 0.0018376, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H1: qlike_reduction_percent', 'report_literal': '+0.9302', 'report_value': 0.9302, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H2: N_origins', 'report_literal': '26898', 'report_value': 26898.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H2: estimate', 'report_literal': '-0.0017400', 'report_value': -0.00174, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AAPL T H2: qlike_reduction_percent', 'report_literal': '-0.8890', 'report_value': -0.889, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AAPL'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H1: N_origins', 'report_literal': '26743', 'report_value': 26743.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H1: estimate', 'report_literal': '+0.0022447', 'report_value': 0.0022447, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H1: qlike_reduction_percent', 'report_literal': '+1.2519', 'report_value': 1.2519, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H2: N_origins', 'report_literal': '26743', 'report_value': 26743.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H2: estimate', 'report_literal': '+0.0005139', 'report_value': 0.0005139, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN R H2: qlike_reduction_percent', 'report_literal': '+0.2903', 'report_value': 0.2903, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H1: N_origins', 'report_literal': '26743', 'report_value': 26743.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H1: estimate', 'report_literal': '+0.0025437', 'report_value': 0.0025437, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H1: qlike_reduction_percent', 'report_literal': '+1.3838', 'report_value': 1.3838, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H2: N_origins', 'report_literal': '26743', 'report_value': 26743.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H2: estimate', 'report_literal': '-0.0000682', 'report_value': -6.82e-05, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'AMZN T H2: qlike_reduction_percent', 'report_literal': '-0.0376', 'report_value': -0.0376, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_AMZN'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H1: N_origins', 'report_literal': '26853', 'report_value': 26853.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H1: estimate', 'report_literal': '+0.0024773', 'report_value': 0.0024773, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H1: qlike_reduction_percent', 'report_literal': '+1.2786', 'report_value': 1.2786, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H2: N_origins', 'report_literal': '26853', 'report_value': 26853.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H2: estimate', 'report_literal': '+0.0000453', 'report_value': 4.53e-05, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META R H2: qlike_reduction_percent', 'report_literal': '+0.0237', 'report_value': 0.0237, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H1: N_origins', 'report_literal': '26853', 'report_value': 26853.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H1: estimate', 'report_literal': '+0.0024923', 'report_value': 0.0024923, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H1: qlike_reduction_percent', 'report_literal': '+1.2558', 'report_value': 1.2558, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H2: N_origins', 'report_literal': '26853', 'report_value': 26853.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H2: estimate', 'report_literal': '+0.0002321', 'report_value': 0.0002321, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'META T H2: qlike_reduction_percent', 'report_literal': '+0.1185', 'report_value': 0.1185, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_META'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H1: N_origins', 'report_literal': '26729', 'report_value': 26729.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H1: estimate', 'report_literal': '+0.0008978', 'report_value': 0.0008978, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H1: qlike_reduction_percent', 'report_literal': '+0.5032', 'report_value': 0.5032, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H2: N_origins', 'report_literal': '26729', 'report_value': 26729.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H2: estimate', 'report_literal': '+0.0007797', 'report_value': 0.0007797, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT R H2: qlike_reduction_percent', 'report_literal': '+0.4392', 'report_value': 0.4392, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H1: N_origins', 'report_literal': '26729', 'report_value': 26729.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H1: estimate', 'report_literal': '+0.0014329', 'report_value': 0.0014329, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H1: qlike_reduction_percent', 'report_literal': '+0.7815', 'report_value': 0.7815, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H2: N_origins', 'report_literal': '26729', 'report_value': 26729.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H2: estimate', 'report_literal': '-0.0006875', 'report_value': -0.0006875, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'MSFT T H2: qlike_reduction_percent', 'report_literal': '-0.3779', 'report_value': -0.3779, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_MSFT'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H1: N_origins', 'report_literal': '26764', 'report_value': 26764.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H1: estimate', 'report_literal': '+0.0006938', 'report_value': 0.0006938, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H1: qlike_reduction_percent', 'report_literal': '+0.3832', 'report_value': 0.3832, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H2: N_origins', 'report_literal': '26764', 'report_value': 26764.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H2: estimate', 'report_literal': '+0.0022787', 'report_value': 0.0022787, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA R H2: qlike_reduction_percent', 'report_literal': '+1.2636', 'report_value': 1.2636, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H1: N_origins', 'report_literal': '26764', 'report_value': 26764.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H1: estimate', 'report_literal': '+0.0017841', 'report_value': 0.0017841, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H1: qlike_reduction_percent', 'report_literal': '+0.9761', 'report_value': 0.9761, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H2: N_origins', 'report_literal': '26764', 'report_value': 26764.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H2: estimate', 'report_literal': '+0.0010290', 'report_value': 0.001029, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'NVDA T H2: qlike_reduction_percent', 'report_literal': '+0.5686', 'report_value': 0.5686, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_NVDA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H1: N_origins', 'report_literal': '26845', 'report_value': 26845.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H1: estimate', 'report_literal': '+0.0009139', 'report_value': 0.0009139, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H1: qlike_reduction_percent', 'report_literal': '+0.5226', 'report_value': 0.5226, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H2: N_origins', 'report_literal': '26845', 'report_value': 26845.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H2: estimate', 'report_literal': '+0.0016557', 'report_value': 0.0016557, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA R H2: qlike_reduction_percent', 'report_literal': '+0.9518', 'report_value': 0.9518, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H1: N_origins', 'report_literal': '26845', 'report_value': 26845.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H1: estimate', 'report_literal': '+0.0030892', 'report_value': 0.0030892, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H1: qlike_reduction_percent', 'report_literal': '+1.7111', 'report_value': 1.7111, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H2: N_origins', 'report_literal': '26845', 'report_value': 26845.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H2: estimate', 'report_literal': '-0.0000429', 'report_value': -4.29e-05, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 19', 'claim': 'TSLA T H2: qlike_reduction_percent', 'report_literal': '-0.0242', 'report_value': -0.0242, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/robustness.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1', 'subset': 'asset_TSLA'}, 'column': 'qlike_reduction_percent', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AAPL: scheduled_rows', 'report_literal': '27,055', 'report_value': 27055.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AAPL'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AAPL: eligible_rows', 'report_literal': '26,898', 'report_value': 26898.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AAPL'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AAPL: invalid_target', 'report_literal': '7', 'report_value': 7.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AAPL'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AAPL: incomplete_mandatory_predictors', 'report_literal': '155', 'report_value': 155.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AAPL'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AAPL: eligible share', 'report_literal': '99.4', 'report_value': 99.4, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AAPL'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'AAPL: known-empty 5m', 'report_literal': '0.26', 'report_value': 0.26, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'AAPL', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'AAPL: known-empty 30m', 'report_literal': '0.24', 'report_value': 0.24, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'AAPL', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'AMZN: scheduled_rows', 'report_literal': '27,055', 'report_value': 27055.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AMZN'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AMZN: eligible_rows', 'report_literal': '26,743', 'report_value': 26743.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AMZN'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AMZN: invalid_target', 'report_literal': '0', 'report_value': 0.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AMZN'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AMZN: incomplete_mandatory_predictors', 'report_literal': '312', 'report_value': 312.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AMZN'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'AMZN: eligible share', 'report_literal': '98.8', 'report_value': 98.8, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'AMZN'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'AMZN: known-empty 5m', 'report_literal': '0.25', 'report_value': 0.25, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'AMZN', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'AMZN: known-empty 30m', 'report_literal': '0.24', 'report_value': 0.24, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'AMZN', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'META: scheduled_rows', 'report_literal': '27,054', 'report_value': 27054.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'META'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'META: eligible_rows', 'report_literal': '26,853', 'report_value': 26853.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'META'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'META: invalid_target', 'report_literal': '0', 'report_value': 0.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'META'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'META: incomplete_mandatory_predictors', 'report_literal': '201', 'report_value': 201.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'META'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'META: eligible share', 'report_literal': '99.3', 'report_value': 99.3, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'META'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'META: known-empty 5m', 'report_literal': '0.23', 'report_value': 0.23, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'META', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'META: known-empty 30m', 'report_literal': '0.22', 'report_value': 0.22, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'META', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'MSFT: scheduled_rows', 'report_literal': '27,055', 'report_value': 27055.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'MSFT'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'MSFT: eligible_rows', 'report_literal': '26,729', 'report_value': 26729.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'MSFT'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'MSFT: invalid_target', 'report_literal': '0', 'report_value': 0.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'MSFT'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'MSFT: incomplete_mandatory_predictors', 'report_literal': '326', 'report_value': 326.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'MSFT'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'MSFT: eligible share', 'report_literal': '98.8', 'report_value': 98.8, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'MSFT'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'MSFT: known-empty 5m', 'report_literal': '0.24', 'report_value': 0.24, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'MSFT', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'MSFT: known-empty 30m', 'report_literal': '0.22', 'report_value': 0.22, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'MSFT', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'NVDA: scheduled_rows', 'report_literal': '27,055', 'report_value': 27055.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'NVDA'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'NVDA: eligible_rows', 'report_literal': '26,764', 'report_value': 26764.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'NVDA'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'NVDA: invalid_target', 'report_literal': '0', 'report_value': 0.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'NVDA'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'NVDA: incomplete_mandatory_predictors', 'report_literal': '291', 'report_value': 291.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'NVDA'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'NVDA: eligible share', 'report_literal': '98.9', 'report_value': 98.9, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'NVDA'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'NVDA: known-empty 5m', 'report_literal': '0.29', 'report_value': 0.29, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'NVDA', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'NVDA: known-empty 30m', 'report_literal': '0.27', 'report_value': 0.27, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'NVDA', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'TSLA: scheduled_rows', 'report_literal': '27,055', 'report_value': 27055.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'TSLA'}, 'column': 'scheduled_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'TSLA: eligible_rows', 'report_literal': '26,845', 'report_value': 26845.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'TSLA'}, 'column': 'eligible_rows', 'operation': None}, {'report_location': 'Table 3', 'claim': 'TSLA: invalid_target', 'report_literal': '7', 'report_value': 7.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'TSLA'}, 'column': 'invalid_target', 'operation': None}, {'report_location': 'Table 3', 'claim': 'TSLA: incomplete_mandatory_predictors', 'report_literal': '203', 'report_value': 203.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'TSLA'}, 'column': 'incomplete_mandatory_predictors', 'operation': None}, {'report_location': 'Table 3', 'claim': 'TSLA: eligible share', 'report_literal': '99.2', 'report_value': 99.2, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'TSLA'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'TSLA: known-empty 5m', 'report_literal': '0.27', 'report_value': 0.27, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'TSLA', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'TSLA: known-empty 30m', 'report_literal': '0.24', 'report_value': 0.24, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'TSLA', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'All: scheduled_rows', 'report_literal': '162,329', 'report_value': 162329.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'All'}, 'column': 'scheduled_rows', 'operation': 'sum assets'}, {'report_location': 'Table 3', 'claim': 'All: eligible_rows', 'report_literal': '160,832', 'report_value': 160832.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'All'}, 'column': 'eligible_rows', 'operation': 'sum assets'}, {'report_location': 'Table 3', 'claim': 'All: invalid_target', 'report_literal': '14', 'report_value': 14.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'All'}, 'column': 'invalid_target', 'operation': 'sum assets'}, {'report_location': 'Table 3', 'claim': 'All: incomplete_mandatory_predictors', 'report_literal': '1488', 'report_value': 1488.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'All'}, 'column': 'incomplete_mandatory_predictors', 'operation': 'sum assets'}, {'report_location': 'Table 3', 'claim': 'All: eligible share', 'report_literal': '99.1', 'report_value': 99.1, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_v4_b4/coverage.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'asset': 'All'}, 'column': None, 'operation': '100*eligible_rows/scheduled_rows; sum first if All'}, {'report_location': 'Table 3', 'claim': 'All: known-empty 5m', 'report_literal': '0.26', 'report_value': 0.26, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'All', 'horizon': '5m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Table 3', 'claim': 'All: known-empty 30m', 'report_literal': '0.24', 'report_value': 0.24, 'unit': 'percentage points', 'tolerance': 0.005, 'source': 'artifacts/rp4_v4_b2_rv15/summary.json#empty_window_secondary.census', 'filter': {'asset': 'All', 'horizon': '30m'}, 'column': None, 'operation': '100*N_empty_origins/N_origins; sum first if All'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: count', 'report_literal': '50', 'report_value': 50.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_count'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: observed', 'report_literal': '0.001134', 'report_value': 0.001134, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_observed'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: mean', 'report_literal': '0.000926', 'report_value': 0.000926, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_mean'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: quantile_025', 'report_literal': '0.000456', 'report_value': 0.000456, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_quantile_025'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: quantile_975', 'report_literal': '0.001283', 'report_value': 0.001283, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_quantile_975'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: exceedances', 'report_literal': '12', 'report_value': 12.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_exceedances'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: ascending_rank', 'report_literal': '39', 'report_value': 39.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_ascending_rank'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV15 placebo: empirical_p', 'report_literal': '0.255', 'report_value': 0.255, 'unit': 'probability', 'tolerance': 0.0005, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv', 'filter': {'horizon': 15}, 'column': 'delta', 'operation': 'placebo_empirical_p'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: count', 'report_literal': '50', 'report_value': 50.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_count'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: observed', 'report_literal': '0.0008024', 'report_value': 0.0008024, 'unit': 'QLIKE', 'tolerance': 5e-08, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_observed'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: mean', 'report_literal': '0.0012107', 'report_value': 0.0012107, 'unit': 'QLIKE', 'tolerance': 5e-08, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_mean'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: exceedances', 'report_literal': '38', 'report_value': 38.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_exceedances'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: ascending_rank', 'report_literal': '13', 'report_value': 13.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_ascending_rank'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: empirical_p', 'report_literal': '0.7647', 'report_value': 0.7647, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_empirical_p'}, {'report_location': 'Section 4.10 / Figure 16', 'claim': 'RV30 placebo: retained_percent', 'report_literal': '150.9', 'report_value': 150.9, 'unit': 'percentage points', 'tolerance': 0.05, 'source': 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv', 'filter': {'horizon': 30}, 'column': 'delta', 'operation': 'placebo_retained_percent'}]

def build_claims_ledger(report_claims, read_public_csv, fetch_bytes, checks, quick=False):
    """Compare fixed report values with verified files and the calculated primary contrasts."""
    import html
    import json
    import math
    import numpy as np
    import pandas as pd
    from IPython.display import HTML, display

    primary_path = 'artifacts/rp4_v4_b4/primary_statistics.csv'
    summary_path = 'artifacts/rp4_v4_b2_rv15/summary.json'
    primary = read_public_csv(primary_path)
    cache = {primary_path: primary}

    def read_rows(source):
        if source not in cache:
            path, _, key = source.partition('#')
            if key:
                value = json.loads(fetch_bytes(path))
                for name in key.split('.'):
                    value = value[name]
                cache[source] = pd.DataFrame(value)
            else:
                cache[source] = read_public_csv(path)
        return cache[source]

    def select(frame, filters):
        chosen = frame
        for name, value in filters.items():
            chosen = chosen.loc[chosen[name].eq(value)]
        return chosen

    def one(frame, filters):
        chosen = select(frame, filters)
        assert len(chosen) == 1, f'Expected one result for {filters}; found {len(chosen)}'
        return chosen.iloc[0]

    def primary_row(horizon, family='log_ridge_harq', contrast='B2_over_B1'):
        return one(primary, {'horizon_minutes': horizon, 'window': 'primary',
                             'family': family, 'contrast': contrast})

    # Check the separately published segment table against its saved summary.
    if not quick:
        segment_rows = read_rows('artifacts/rp4_v4_b4/regime_secondary.csv')
        summary_rows = read_rows(summary_path + '#regime_secondary')
        for row in select(segment_rows, {'horizon_minutes': 15, 'window': 'primary',
                                         'statistic': 'mean'}).itertuples():
            if row.subset not in ('first_hour', 'last_hour'):
                continue
            saved = one(summary_rows, {'subset': row.subset, 'family': row.family,
                                       'contrast': row.contrast, 'statistic': 'mean'})
            for name in ('estimate', 'ci_low', 'ci_high', 'p_raw', 'p_holm'):
                assert abs(float(saved[name]) - float(getattr(row, name))) <= 1e-12

    results = []
    selected_claims = [c for c in report_claims
                       if c['source'] != 'NOT_IN_12_CSV'
                       and (not quick or c['report_location'] in
                            ('Table 7', 'Table 11', 'Table 12', 'Table 13'))]
    assert selected_claims, 'The claims list is empty.'
    for claim in selected_claims:
        filters = dict(claim['filter'])
        operation = claim.get('operation')
        source = claim['source']
        column = claim['column']
        basis = 'Saved statistic verified'
        if claim['report_location'] == 'Table 7' and column in ('estimate', 'percent_reduction_mean'):
            row = one(checks, {'family': filters['family'], 'contrast': filters['contrast']})
            actual = float(row['delta_QLIKE' if column == 'estimate' else 'reduction_percent'])
            basis = 'Paired losses recalculated'
        elif operation == 'favourable_sessions':
            rows = read_rows(source)
            assert len(rows) == rows.session_date.nunique() == 419
            baseline_column = 'loss__log_ridge_harq__B1' if 'loss__log_ridge_harq__B1' in rows else 'B1'
            expanded_column = 'loss__log_ridge_harq__B2' if 'loss__log_ridge_harq__B2' in rows else 'B2'
            paired = rows[[baseline_column, expanded_column]].to_numpy(dtype=float)
            assert np.isfinite(paired).all()
            actual = int((paired[:, 0] > paired[:, 1]).sum())
            basis = 'Paired favourable sessions recalculated; 419 verified sessions'
        elif operation and operation.startswith('placebo_'):
            horizon = filters['horizon']
            draws = select(read_rows(source), filters)['delta'].to_numpy(dtype=float)
            assert len(draws) == 50 and np.isfinite(draws).all()
            observed = float(primary_row(horizon).estimate)
            operation_name = operation[len('placebo_'):]
            actual = {
                'count': len(draws), 'observed': observed, 'mean': math.fsum(draws) / len(draws),
                'quantile_025': float(np.quantile(draws, .025)),
                'quantile_975': float(np.quantile(draws, .975)),
                'exceedances': int((draws > observed).sum()),
                'ascending_rank': 1 + int((draws < observed).sum()),
                'empirical_p': (1 + int((draws >= observed).sum())) / (len(draws) + 1),
                'retained_percent': 100 * (math.fsum(draws) / len(draws)) / observed,
            }[operation_name]
            basis = 'Saved permutation draws summarised'
        elif claim['report_location'] == 'Table 17' and operation:
            row = one(read_rows(source), filters)
            state_filter = dict(filters, contrast='B1_over_B0')
            state = one(read_rows(source), state_filter)
            baseline = float(primary_row(15, contrast='B1_over_B0').baseline_loss)
            if filters['contrast'] == 'B2_over_B1':
                baseline -= float(state.estimate)
            assert baseline > 0
            actual = 100 * float(row.estimate) / baseline
            basis = 'Saved cutoff difference rescaled'
        elif claim['report_location'] == 'Table 3':
            all_assets = filters.get('asset') == 'All'
            if all_assets:
                filters.pop('asset')
            rows = select(read_rows(source), filters)
            assert len(rows) == (6 if all_assets else 1)
            if column:
                actual = float(rows[column].sum())
            elif source.endswith('#empty_window_secondary.census'):
                actual = 100 * rows.N_empty_origins.sum() / rows.N_origins.sum()
            else:
                actual = 100 * rows.eligible_rows.sum() / rows.scheduled_rows.sum()
            basis = 'Saved census counts aggregated'
        else:
            row = one(read_rows(source), filters)
            actual = row[column]
            if operation == 'status_label':
                actual = {'REJECTED': 'Reject null', 'NOT_REJECTED': 'Do not reject',
                          'NOT_TESTED': 'Gate closed'}[actual]
        expected = claim['report_value']
        tolerance = claim['tolerance']
        if claim['unit'] == 'status':
            matched = actual == expected
            computed = str(actual)
            allowed = 'Exact state'
        else:
            actual = float(actual)
            assert math.isfinite(actual) and math.isfinite(float(expected))
            matched = abs(actual - expected) <= tolerance
            computed = f'{actual:.10g}'
            allowed = f'{tolerance:g} {claim["unit"]}'
            if operation == 'favourable_sessions':
                computed = f'{int(actual)}/419'
        assert matched, f'{claim["report_location"]}, {claim["claim"]}: {actual} differs from {expected}'
        results.append({'Report location': claim['report_location'], 'Claim': claim['claim'],
                        'Report value': claim['report_literal'], 'Recomputed here': computed,
                        'Tolerance': allowed, 'Basis': basis, 'Match': '✔'})
    ledger = pd.DataFrame(results)
    assert len(ledger) == len(selected_claims) and ledger['Match'].eq('✔').all()
    # Group the complete ledger into expandable tables for a compact reading view.
    parts = [f'<p><strong>Claims ledger: {len(ledger)} of {len(ledger)} matched ✔</strong></p>']
    for location, group in ledger.groupby('Report location', sort=False):
        parts.append('<details><summary>' + html.escape(location) + f': {len(group)} matched</summary>'
                     + group.drop(columns='Report location').to_html(index=False, escape=True)
                     + '</details>')
    display(HTML(''.join(parts)))
    return ledger

# Add the three fixed report counts; their denominators are checked from the rows.
REPORT_CLAIMS.extend([{'report_location': 'Table 17', 'claim': 'Linear H2 favourable sessions, 60 s', 'report_value': 277, 'report_literal': '277/419', 'source': 'artifacts/rp4_robustness_public_v1/pit_60_session_losses.csv', 'filter': {}, 'column': None, 'operation': 'favourable_sessions', 'unit': 'sessions', 'tolerance': 0}, {'report_location': 'Table 17', 'claim': 'Linear H2 favourable sessions, 120 s', 'report_value': 249, 'report_literal': '249/419', 'source': 'artifacts/rp4_v4_b2_rv15/session_losses.csv', 'filter': {}, 'column': None, 'operation': 'favourable_sessions', 'unit': 'sessions', 'tolerance': 0}, {'report_location': 'Table 17', 'claim': 'Linear H2 favourable sessions, 300 s', 'report_value': 217, 'report_literal': '217/419', 'source': 'artifacts/rp4_robustness_public_v1/pit_300_session_losses.csv', 'filter': {}, 'column': None, 'operation': 'favourable_sessions', 'unit': 'sessions', 'tolerance': 0}])

# Stop on the first mismatch before allowing the receipt to run.
claims_ledger = build_claims_ledger(REPORT_CLAIMS, read_public_csv, fetch_bytes, checks, quick=False)

# Record that this step completed.
EXECUTED_CELLS.append('step_21_claims_ledger')


Claim,Report value,Recomputed here,Tolerance,Basis,Match
Linear H1: estimate,+0.002547,0.002546895918,1e-06 QLIKE,Saved statistic verified,✔
Linear H1: ci_low,+0.000557,0.0005570052728,1e-06 QLIKE,Saved statistic verified,✔
Linear H1: ci_high,+0.005541,0.005540719607,1e-06 QLIKE,Saved statistic verified,✔
Linear H1: percent_reduction_mean,+1.729,1.728974456,0.0005 percentage points,Saved statistic verified,✔
Linear H1: p_raw,0.0439,0.0439,5e-05 probability,Saved statistic verified,✔
Linear H1: decision,Reject null,Reject null,Exact state,Saved statistic verified,✔
Linear H2: estimate,+0.000802,0.0008023896595,1e-06 QLIKE,Saved statistic verified,✔
Linear H2: ci_low,-0.000153,-0.0001534346958,1e-06 QLIKE,Saved statistic verified,✔
Linear H2: ci_high,+0.001774,0.001773543683,1e-06 QLIKE,Saved statistic verified,✔
Linear H2: percent_reduction_mean,+0.554,0.5542902183,0.0005 percentage points,Saved statistic verified,✔


### Reading

Every checked row must show ✔. A saved probability is verified against the report, while the four Table 7 means and reductions are recalculated from paired session losses. The ledger therefore states which kind of evidence supports each row.


## What this notebook establishes

The covered values in Tables 3, 6 to 14, 17 and 19 agree with the published aggregates within the stated tolerances. The input checks fail if any byte changes; the ledger stops if a checked value differs from the report.

## What it cannot establish

Model fitting, predictor construction and point-in-time filters run on licensed minute-level data that cannot be redistributed. Their outputs enter this notebook as saved aggregates. Section 3.10 explains what is retained and how access works. The historical final window does not confirm the full sequence in Table 13.

## Receipt

The final cell checks the exact step order and records the input fingerprints, numerical contrasts and software versions. A completed run shows PASS_PUBLIC_SAVED_RESULTS. It records the verification supporting Table 7 and the Claims ledger.


In [22]:
# @title Receipt
receipt = {'status':'PASS_PUBLIC_SAVED_RESULTS', 'source_commit':SOURCE_COMMIT,
    'public_source_commit':PUBLIC_SOURCE_COMMIT,
    'bundle_sha256':BUNDLE_SHA256, 'input_sha256':PUBLIC_HASHES,
    'sessions':len(losses), 'contrasts_recalculated':checks.to_dict('records'),
    'public_reference_rows':len(reference_qlike),
    'public_reference_contrast_rows':len(reference_contrasts),
    'secondary_metric_rows':len(secondary_metrics), 'new_fits':0,
    'restricted_files_read':0, 'widgets':0,
    'scope':'Public saved-results arithmetic and static figures; no training or new inference',
    'runtime':{'python':sys.version.split()[0], 'pandas':pd.__version__, 'matplotlib':matplotlib.__version__}}

# Require every preceding step, including the ledger, to have completed.
EXECUTED_CELLS.append('step_22_receipt')
assert EXECUTED_CELLS == ['step_01_verify_inputs', 'step_02_three_example_sessions', 'step_03_the_four_primary_contrasts', 'step_04_the_final_window', 'step_05_coverage_of_the_development_sample', 'step_06_session_differences_assets_and_chronological_blocks', 'step_07_reference_forecasts', 'step_08_complementary_errors', 'step_09_complete_saved_results', 'step_10_a_second_implementation', 'step_11_check_the_values_represented', 'step_12_figure_5_coverage_and_known_empty_windows', 'step_13_figure_6_primary_contrasts_and_saved_intervals', 'step_14_figures_7_and_8_session_distributions_and_cumulative_paths', 'step_15_table_9_session_segments', 'step_16_table_10_volatility_regimes', 'step_17_table_14_inference_sensitivity', 'step_18_table_17_availability_cutoffs', 'step_19_figure_15_shuffled_mixed_flow_comparisons', 'step_20_figure_11_subset_comparisons', 'step_21_claims_ledger', 'step_22_receipt']
receipt['executed_code_cells']=len(EXECUTED_CELLS)
receipt['executed_code_cell_ids']=EXECUTED_CELLS.copy()
receipt['supplementary_bundle_sha256'] = SUPPLEMENTARY_SHA256
receipt_json=json.dumps(receipt,indent=2,ensure_ascii=False)
Path('PUBLIC_EXECUTION_RECEIPT.json').write_text(receipt_json,encoding='utf-8')
# Present a short result; retain the complete machine-readable record below.
display(HTML('<h3>Verification complete</h3><p><b>PASS_PUBLIC_SAVED_RESULTS</b></p><p>'+str(len(EXECUTED_CELLS))+' code cells completed; '+str(len(claims_ledger))+' report values matched.</p><details><summary>Complete execution receipt</summary><pre>'+html.escape(receipt_json)+'</pre></details>'))

